# PersonaPlex + IMTalker Emotion Live Avatar Server — RunPod RTX 5090 Deployment

This notebook deploys the **emotion-conditioned PersonaPlex + IMTalker AH AudioPace
live server** on the new pipeline. It extends `runpod_emotion_speech2face.ipynb`
with a trained **emotion LoRA adapter** so the avatar keeps one consistent facial
expression for the entire conversation.

**Auto-Emotion Mode (new):** Set `AUTO_EMOTION_DETECTION = True` in the
Parameters cell to enable automatic per-utterance emotion detection. After
the server is running, execute **Step 19** (install SER model) and **Step 20**
(start the auto-emotion worker). The worker listens to user audio, runs a
Speech Emotion Recognition model (wav2vec2-based, ~20 ms/utterance on the
RTX 5090) after each user utterance, and calls `POST /api/emotion` on the
live server so the avatar's facial expression updates dynamically — no server
restart required.

**Manual mode (original):** Set `AUTO_EMOTION_DETECTION = False` and select
the emotion in the Parameters cell. The expression is then fixed for the
entire conversation.

**Select the emotion before launching (manual mode)** (Parameters cell): `happy`, `sad`,
`angry`, `surprise`, `fear` (or `neutral`). The choice is baked into the GPU
weights at startup and never changes mid-conversation.

**Pipeline:**
```
PersonaPlex assistant audio + hidden states (12.5 Hz)
-> assistant-output RMS gate (silence Helium seed while not replying)
-> 0.96s lookahead adapter (unitalk_last_layer, 12 hidden steps + 6 future steps)
-> IMTalker generator: base + static-head style LoRA (merged) + emotion LoRA (fixed emotion_id)
-> IMTalker FP32 renderer
-> separate Opus audio (/ws/conversation) + JPEG video (/ws/video) websockets
```

**How the emotion integration works (new pipeline):**
1. The stock new-pipeline repo ships without emotion conditioning, so **Step 12**
   injects the exact same emotion code the adapter was trained with
   (`EmotionConditioner` in `generator/FM.py`, emotion-bias kwargs in
   `generator/FMT.py`, `apply_emotion_lora_to_model` in `generator/train_lora.py`)
   — byte-identical to `train_emotion_lora.ipynb`, and idempotent.
2. Step 12 also patches the live server (`--emotion_*` args, emotion-aware
   `_load_fm`, fixed `emotion_id` in every `fm.sample()` call) and
   `start_winner_live.sh` (`EMOTION_ARGS` built from env vars).
3. At load time the server merges the 2h static-head style LoRA (rank 64) into
   the base generator weights, then applies your emotion LoRA (rank 32) on top
   — the same stacking order used in training.
4. The emotion bias is added to the FMT conditioning sum downstream of the
   Helium audio frontend, so speech generation, audio streaming, lip-sync and
   real-time animation are untouched.
5. **Expression strength (compositional CFG).** The adapter was trained/validated
   at `a_cfg_scale=3.0`, but streaming keeps the audio CFG at the AH lip-sync
   default (~1.13) — which alone is too weak to move the face (the "always
   neutral" bug). Step 12 therefore patches `forward_with_cfg` into a 3-way
   **compositional CFG**: audio guidance stays at `a_cfg_scale` while the emotion
   delta gets its own `EMOTION_CFG_SCALE` (default **3.0**). Tune it in the
   Parameters cell — higher = stronger expression, 1.0 = no extra emotion guidance.

**Emotion adapter:** `bitwisemind/emotion_adapter_v1` (`best.ckpt`, EMA weights)

> Run this top-to-bottom on a fresh RunPod RTX 5090 pod (Ubuntu, CUDA 12.8 base image).
> Edit the **Parameters** cell below (especially `EMOTION`) before launching.


In [ ]:
!git clone https://github.com/MoshiHead/IMTalker_Personaplex_emotion_v1_auto_emotion_anti_v1.git speech2avatar

## Parameters

Edit these before running. Leaving an override blank means "use the default
baked into `run_live.sh`" — this notebook never edits that script, it only sets
environment variables that the script already reads.


In [ ]:
import os

# --- Project location -------------------------------------------------------
# If PROJECT_ROOT already contains a valid speech2avatar checkout, it is used
# as-is. Otherwise, if GIT_REPO_URL is set, the repo is cloned into PROJECT_ROOT.
PROJECT_ROOT = "/workspace/speech2avatar"
GIT_REPO_URL = "https://github.com/nash-raf/speech2avatar.git"  # leave blank if already cloned (e.g. by the cell above)
GIT_BRANCH = "main"

# --- Toolchain ---------------------------------------------------------------
VENV_DIR = "/workspace/preprocess_5090"

# --- Service ------------------------------------------------------------------
# PORT must exactly match an "HTTP Port" / "TCP Port" exposed in your RunPod pod
# template. RunPod's proxy (https://<pod-id>-<port>.proxy.runpod.net/) only
# forwards traffic for ports that were actually exposed when the pod was
# created/edited — if the server binds a port your pod doesn't expose, the
# proxy URL will hang or refuse the connection even though the server is
# healthy. Check your pod's "Connect" -> "HTTP Service" list on the RunPod
# dashboard and set PORT to one of those values (commonly 8998 or 8999 for
# this stack; AH's own standalone default is 8998, run_live.sh's own default
# is 8999 if PORT is left unset).
HOST = "0.0.0.0"  # informational only: start_winner_live.sh always binds 0.0.0.0
PORT = 8998

# RunPod sets RUNPOD_POD_ID automatically inside every pod; used only to print
# the exact public proxy URL below. Override manually if it's not auto-detected
# (e.g. running this notebook outside of a RunPod pod).
RUNPOD_POD_ID = os.environ.get("RUNPOD_POD_ID", "")
PUBLIC_URL = f"https://{RUNPOD_POD_ID}-{PORT}.proxy.runpod.net/" if RUNPOD_POD_ID else None

# --- EMOTION — select the avatar's expression for the whole conversation ------
# Valid values: "happy", "sad", "angry", "surprise", "fear" (plus "neutral").
# The selected emotion is fixed at launch time and stays consistent for the
# entire streaming session. To change it: run the stop cell, edit EMOTION here,
# re-run this cell, then re-run the Launch cell (Step 16).
EMOTION = "happy"

EMOTION_MAP = {"neutral": 0, "happy": 1, "sad": 2, "angry": 3, "surprise": 4, "fear": 5}
EMOTION_NAMES = {v: k for k, v in EMOTION_MAP.items()}
if EMOTION.lower() not in EMOTION_MAP:
    raise ValueError(f"Invalid EMOTION={EMOTION!r}; choose one of {sorted(EMOTION_MAP)}")
EMOTION_ID = EMOTION_MAP[EMOTION.lower()]

# Trained emotion adapter on Hugging Face (full Lightning checkpoint, EMA weights).
HF_EMOTION_ADAPTER_REPO = "bitwisemind/emotion_adapter_v1"
HF_EMOTION_ADAPTER_FILE = "best.ckpt"
# LoRA hyper-parameters used at training time (must match train_emotion_lora.ipynb).
EMOTION_LORA_RANK = 32
EMOTION_LORA_ALPHA = 64.0
# Compositional CFG scale for the EMOTION delta only. The adapter was trained and
# validated at a_cfg_scale=3.0, but streaming keeps the audio CFG at the AH lip-sync
# default (~1.13). The patched forward_with_cfg gives the emotion delta its own
# guidance scale so the expression is visible without over-amplifying the audio
# (lip-sync) conditioning. 3.0 matches the validated strength; raise for a stronger
# expression, lower toward 1.0 for a subtler one. 1.0 = no extra emotion guidance.
EMOTION_CFG_SCALE = 3.0

# --- AUTO EMOTION DETECTION ---------------------------------------------------
# When True, a Speech Emotion Recognition (SER) model automatically detects the
# user's emotion from their voice on every utterance and updates the avatar's
# facial expression in real time (no server restart required).
# Steps 19 and 20 (new cells after Step 18) set up and start the detection
# worker. The startup emotion (EMOTION above) is used until the first utterance
# is classified.
#
# Set to False to keep the original behaviour: a fixed emotion for the whole
# session selected via EMOTION above.
AUTO_EMOTION_DETECTION = True

# Speech Emotion Recognition model (HuggingFace model ID).
# Accepts any AudioClassification model whose label set includes at least a
# subset of: neutral, happy, sad, angry, surprised/surprise, fear/fearful.
# Runs on the same CUDA device as the FM generator.
SER_MODEL_ID = "r-f/wav2vec-english-speech-emotion-recognition"

# Maps SER output labels (lowercase) → your 6-class EMOTION_MAP keys.
# Add extra synonyms here if the model uses different label names.
SER_LABEL_MAP = {
    # neutral
    "neutral": "neutral", "neutrality": "neutral", "calm": "neutral",
    # happy
    "happy": "happy", "happiness": "happy",
    # sad
    "sad": "sad", "sadness": "sad",
    # angry
    "angry": "angry", "anger": "angry",
    # surprise
    "surprise": "surprise", "surprised": "surprise",
    # fear
    "fear": "fear", "fearful": "fear", "fear_1": "fear",
    # fallback: disgust → neutral (no disgust class in your schema)
    "disgust": "neutral",
}

# Minimum utterance duration (seconds) to run SER — very short blips are ignored.
SER_MIN_UTTERANCE_SEC = 0.6

# RMS level below which audio is considered silence for utterance segmentation.
# Raise if background noise causes false positives; lower for quieter microphones.
SER_SILENCE_RMS_THRESHOLD = 0.015

# Silence duration (seconds) that marks the end of a user utterance.
SER_UTTERANCE_END_SILENCE_SEC = 0.9

# Minimum SER confidence (0–1) required to update the avatar emotion.
# If confidence is below this, the previous emotion is kept.
SER_MIN_CONFIDENCE = 0.35
# Dedicated log file for tracking user voice emotion predictions
AUTO_EMOTION_LOG_PATH = os.path.join(PROJECT_ROOT, "auto_emotion_predictions.log")
AUTO_EMOTION_PID_PATH = "/tmp/auto_emotion_worker.pid"

# How many frames to smooth the emotion over (exponential moving average over
# confidence scores per class). 1 = no smoothing (instant switch on each
# utterance). Higher values make transitions more gradual.
SER_SMOOTHING_WINDOW = 1

# --- Hugging Face sources (must match live.md / live_working.md / winner_latest_1.md) ---
# Live lookahead/RMS adapter + silence-Helium seed (both in the same dataset repo).
HF_ADAPTER_REPO = "niloy629/hdtf_preprocess"
HF_ADAPTER_FILE = "personaplex_lookahead_rms_adapter/checkpoints/personaplex_lookahead096_future048_rms50_adapter.pt"
HF_SILENCE_FILE = "personaplex_lookahead_rms_adapter/stats/silence_helium_mean.pt"

# Current AH static-head with-audio LoRA (2-hour "live_winner" checkpoint).
HF_LORA_REPO = "niloy629/hdtf_preprocess"
HF_LORA_FILE = "live_winner/lora/ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt"
HF_LORA_SHA256 = "f34de1bd748857bc802102da578046bb10bd5f664460607d9c785ff35922e52f"

# Cached blink motion-map (downloaded by download_live_assets.sh; not wired into
# the current AH/AJ default launch, kept here so the asset is still verified).
HF_BLINK_REPO = "niloy629/hdtf_preprocess"
HF_BLINK_FILE = "lora/3robert_audio3_ditto_static_motion.pt"
HF_BLINK_SHA256 = "e29a41ff004b228d7efee15cad0f32f4d4bc5466563709e2ba78b158d4e340bb"

HF_IMTALKER_REPO = "cbsjtu01/IMTalker"
HF_IMTALKER_FILES = [
    "config.yaml",
    "renderer.ckpt",
    "generator.ckpt",
    "wav2vec2-base-960h/config.json",
    "wav2vec2-base-960h/pytorch_model.bin",
    "wav2vec2-base-960h/preprocessor_config.json",
    "wav2vec2-base-960h/feature_extractor_config.json",
]

HF_PERSONAPLEX_REPO = "brianmatzelle/personaplex-7b-v1-bnb-4bit"

# Gated repo: accept the terms for nvidia/personaplex-7b-v1 on Hugging Face with
# the same account as HF_TOKEN before running Step 8, or these downloads 403.
HF_PERSONAPLEX_MIMI_REPO = "nvidia/personaplex-7b-v1"
HF_PERSONAPLEX_MIMI_FILES = [
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "tokenizer_spm_32k_3.model",
]
HF_VOICES_FILE = "voices.tgz"  # extracted to PERSONAPLEX_BNB4_DIR/voices/; VARM3.pt is the live default

EXPECTED_TORCH_VERSION = "2.8.0+cu128"


# --- Optional run_live.sh overrides (leave "" to use the script's own default) ---
REF_PATH = ""            # e.g. f"{PROJECT_ROOT}/IMTalker/assets/source_5.png"
A_CFG_SCALE = ""         # e.g. "1.13" (AH default) or "1.15" (AJ default)
NFE = ""                 # e.g. "5" (script default)
PROMPT_FILE = ""         # e.g. f"{PROJECT_ROOT}/IMTalker/prompts/RB_Robert_System_Prompt_full.txt" (script default)
VOICE_PROMPT = ""        # e.g. "VARM3.pt" (script default; use VARM3 unless explicitly told otherwise)
VOICE_PROMPT_DIR = ""    # e.g. "/workspace/voices"
LORA_GENERATOR_PATH = "" # e.g. path to an alternate static-head LoRA checkpoint
DISABLE_LORA = ""        # e.g. "1" to compare the base generator without the style LoRA (emotion LoRA still applies)
JITTER_BUFFER_SEC = ""   # e.g. "1.0" (script default) seconds of end-to-end A/V jitter buffer; raise to 2-3 for max smoothness


# --- Startup wait behaviour ---------------------------------------------------
STARTUP_TIMEOUT_SEC = 900   # PersonaPlex (7B, 4-bit) + IMTalker model load can take a while
POLL_INTERVAL_SEC = 5

# --- Derived paths (must match download_live_assets.sh / run_live.sh) -------
IMTALKER_DIR = f"{PROJECT_ROOT}/IMTalker"
PERSONAPLEX_DIR = f"{PROJECT_ROOT}/personaplex"
CHECKPOINT_DIR = f"{PROJECT_ROOT}/checkpoints"
PERSONAPLEX_BNB4_DIR = f"{CHECKPOINT_DIR}/personaplex_bnb4"
IMTALKER_CKPT_DIR = f"{IMTALKER_DIR}/checkpoints"
ADAPTER_PATH = f"{CHECKPOINT_DIR}/{HF_ADAPTER_FILE}"
SILENCE_HELIUM_PATH = f"{CHECKPOINT_DIR}/{HF_SILENCE_FILE}"
LORA_PATH = f"{CHECKPOINT_DIR}/{HF_LORA_FILE}"
BLINK_MOTION_PATH = f"{CHECKPOINT_DIR}/{HF_BLINK_FILE}"
PERSONAPLEX_WEIGHT_PATH = f"{PERSONAPLEX_BNB4_DIR}/model_bnb_4bit.pt"
VOICE_PROMPT_PATH = f"{PERSONAPLEX_BNB4_DIR}/voices/{VOICE_PROMPT or 'VARM3.pt'}"
EMOTION_ADAPTER_DIR = f"{CHECKPOINT_DIR}/emotion_adapter"
EMOTION_ADAPTER_PATH = f"{EMOTION_ADAPTER_DIR}/{HF_EMOTION_ADAPTER_FILE}"


VENV_PYTHON = f"{VENV_DIR}/bin/python"
VENV_PIP = f"{VENV_DIR}/bin/pip"
VENV_ACTIVATE = f"source {VENV_DIR}/bin/activate"

LOG_PATH = f"{PROJECT_ROOT}/live_server.log"
PID_PATH = f"{PROJECT_ROOT}/.run_live.pid"

os.makedirs("/workspace", exist_ok=True)
os.makedirs(EMOTION_ADAPTER_DIR, exist_ok=True)
print("Parameters loaded.")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"VENV_DIR     = {VENV_DIR}")
print(f"HOST:PORT    = {HOST}:{PORT}")
print(f"EMOTION      = {EMOTION} (id={EMOTION_ID}) — fixed for the whole session")
print(f"EMOTION ADAPTER = {HF_EMOTION_ADAPTER_REPO}/{HF_EMOTION_ADAPTER_FILE} (rank={EMOTION_LORA_RANK}, alpha={EMOTION_LORA_ALPHA})")
print(f"EMOTION_CFG_SCALE = {EMOTION_CFG_SCALE} (emotion-only guidance; audio CFG stays at the AH default)")
if RUNPOD_POD_ID:
    print(f"RUNPOD_POD_ID = {RUNPOD_POD_ID}")
    print(f"PUBLIC_URL    = {PUBLIC_URL}")
    print(f"  -> Double-check this port is listed under this pod's Connect -> HTTP Service on the RunPod dashboard.")
else:
    print("[warn] RUNPOD_POD_ID not auto-detected (not running inside a RunPod pod, or the env var is unset).")
    print("       Set RUNPOD_POD_ID manually above if you want the public proxy URL printed automatically.")



## Utilities

Shared helpers used throughout the notebook: a streaming shell runner with
retries, a torch/CUDA probe that always queries the **venv** interpreter (the
notebook kernel itself need not have torch installed), a sha256 checksum
helper, a targeted Hugging Face downloader, and port helpers. These back all
of the "Error Recovery" behaviour required by this deployment.


In [ ]:
import hashlib
import json as _json
import socket
import subprocess
import time


def run(cmd, cwd=None, env=None, check=True, retries=1, retry_delay=8, quiet=False):
    last_returncode = None
    for attempt in range(1, retries + 1):
        if not quiet:
            print(f"$ {cmd}" + (f"   [attempt {attempt}/{retries}]" if retries > 1 else ""))
        proc = subprocess.Popen(
            cmd, shell=True, executable="/bin/bash", cwd=cwd,
            env=env or os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
        proc.wait()
        last_returncode = proc.returncode
        if last_returncode == 0:
            return 0
        print(f"[warn] command failed with exit code {last_returncode}")
        if attempt < retries:
            print(f"[recovery] retrying in {retry_delay}s...")
            time.sleep(retry_delay)
    if check:
        raise RuntimeError(f"Command failed after {retries} attempt(s) (exit {last_returncode}): {cmd}")
    return last_returncode


def get_torch_info():
    probe = (
        "import torch, json;"
        "print(json.dumps({"
        "'version': torch.__version__,"
        "'cuda_available': torch.cuda.is_available(),"
        "'cuda_version': torch.version.cuda,"
        "'device_name': (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"
        "}))"
    )
    out = subprocess.run([VENV_PYTHON, "-c", probe], capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stdout)
        print(out.stderr)
        raise RuntimeError("Failed to query torch/CUDA state inside the venv")
    return _json.loads(out.stdout.strip().splitlines()[-1])


def assert_torch_pin(auto_fix=True):
    info = get_torch_info()
    print(f"[info] torch={info['version']} cuda_available={info['cuda_available']} "
          f"cuda_version={info['cuda_version']} device={info['device_name']}")
    if info["version"] != EXPECTED_TORCH_VERSION:
        print(f"[warn] torch version drifted to '{info['version']}' (expected '{EXPECTED_TORCH_VERSION}')")
        if not auto_fix:
            raise RuntimeError(f"torch pin violated: {info['version']} != {EXPECTED_TORCH_VERSION}")
        print("[recovery] reinstalling pinned torch/torchvision/torchaudio (cu128, no-deps)...")
        run(
            f"{VENV_ACTIVATE} && pip install --force-reinstall --no-deps "
            f"torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
            f"--index-url https://download.pytorch.org/whl/cu128",
            retries=3,
        )
        info = get_torch_info()
        if info["version"] != EXPECTED_TORCH_VERSION:
            raise RuntimeError(f"torch pin recovery failed; still got {info['version']}")
    print(f"[ok] torch pin confirmed: {info['version']}")
    return info


def sha256_of(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def hf_download(repo, filename, local_dir, repo_type="model", retries=3):
    repo_type_flag = f"--repo-type {repo_type}" if repo_type != "model" else ""
    os.makedirs(local_dir, exist_ok=True)
    cmd = (
        f"{VENV_ACTIVATE} && export HF_HUB_ENABLE_HF_TRANSFER=1 && "
        f"hf download {repo} {filename} {repo_type_flag} --local-dir {local_dir}"
    )
    run(cmd, retries=retries, retry_delay=15)


def verify_checksum_with_recovery(path, expected_sha256, repo, filename, repo_type, local_dir, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        if not os.path.exists(path):
            print(f"[fix] missing file, downloading: {path} (attempt {attempt}/{max_attempts})")
            hf_download(repo, filename, local_dir, repo_type)
        actual = sha256_of(path)
        if actual.lower() == expected_sha256.lower():
            print(f"[ok] checksum verified: {path}")
            return True
        print(f"[warn] checksum mismatch for {path}")
        print(f"       expected: {expected_sha256}")
        print(f"       actual:   {actual}")
        print(f"[recovery] deleting and re-downloading (attempt {attempt}/{max_attempts})")
        try:
            os.remove(path)
        except FileNotFoundError:
            pass
        hf_download(repo, filename, local_dir, repo_type)
    raise RuntimeError(f"Checksum verification failed after {max_attempts} attempts: {path}")


def port_listening(host, port, timeout=1.0):
    probe_host = "127.0.0.1" if host == "0.0.0.0" else host
    try:
        with socket.create_connection((probe_host, port), timeout=timeout):
            return True
    except OSError:
        return False


def find_pids_on_port(port):
    out = subprocess.run(
        f"fuser {port}/tcp 2>/dev/null || lsof -t -i:{port} 2>/dev/null",
        shell=True, executable="/bin/bash", capture_output=True, text=True,
    )
    return [p for p in out.stdout.split() if p.strip().isdigit()]


def tail_log(n=200):
    if not os.path.exists(LOG_PATH):
        print(f"[info] no log file yet at {LOG_PATH}")
        return
    with open(LOG_PATH, "r", errors="ignore") as f:
        lines = f.readlines()
    print("".join(lines[-n:]))


print("Utilities loaded.")


def tail_auto_emotion_log(lines=40):
    """Print the last N lines of the auto-emotion prediction log."""
    if not os.path.exists(AUTO_EMOTION_LOG_PATH):
        print(f"[info] no auto-emotion log found at {AUTO_EMOTION_LOG_PATH}")
        return
    with open(AUTO_EMOTION_LOG_PATH, "r", encoding="utf-8", errors="replace") as f:
        content = f.readlines()
    print("".join(content[-lines:]))


## Step 0 — Validate / clone the project structure, fix script permissions

`live.md` / `winner_latest_1.md` expect the layout:

```
speech2avatar/
  IMTalker/
    start_winner_live.sh
    start_typeah_audio_pace_varm3.sh
    liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py
    prompts/RB_Robert_System_Prompt_full.txt
    static/index_v3_binary_fullscreen_aj_nodrop.html
  personaplex/
  scripts/download_live_assets.sh
  run_live.sh
  live.md
```

`winner_latest_1.md` explicitly warns that the AH-winner files were, at one
point, only present in a local working tree and might be missing from a
plain clone of the GitHub remote — so this check verifies them by name
instead of assuming a successful `git clone` is sufficient. If `PROJECT_ROOT`
does not already contain this layout and `GIT_REPO_URL` is set, the repo is
cloned. Otherwise the cell fails fast with a clear diagnostic rather than
proceeding against a broken layout.

This step also runs `chmod +x` on every `*.sh` file in the checkout. In
practice, a clone of this repo can arrive with `IMTalker/start_winner_live.sh`
missing its executable bit; since `start_typeah_audio_pace_varm3.sh` invokes
it with `exec "$SCRIPT_DIR/start_winner_live.sh"` (not `bash ...`), a missing
`+x` bit makes Step 14 fail immediately with `Permission denied` and the live
server process exiting with code 126 — before any Python/CUDA/model code
ever runs.


In [ ]:
REQUIRED_STRUCTURE = [
    "IMTalker",
    "personaplex",
    "scripts/download_live_assets.sh",
    "run_live.sh",
    "live.md",
    "IMTalker/start_winner_live.sh",
    "IMTalker/start_typeah_audio_pace_varm3.sh",
    "IMTalker/liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py",
    "IMTalker/prompts/RB_Robert_System_Prompt_full.txt",
    "IMTalker/static/index_v3_binary_fullscreen_aj_nodrop.html",
]


def structure_ok(root):
    return all(os.path.exists(os.path.join(root, rel)) for rel in REQUIRED_STRUCTURE)


if not structure_ok(PROJECT_ROOT):
    if GIT_REPO_URL:
        print(f"[fix] {PROJECT_ROOT} missing or incomplete; cloning {GIT_REPO_URL} (branch {GIT_BRANCH})")
        os.makedirs(os.path.dirname(PROJECT_ROOT.rstrip("/")) or "/", exist_ok=True)
        if os.path.exists(PROJECT_ROOT) and not os.listdir(PROJECT_ROOT):
            os.rmdir(PROJECT_ROOT)
        run(f"git clone --branch {GIT_BRANCH} {GIT_REPO_URL} {PROJECT_ROOT}", retries=3, retry_delay=10)
    else:
        raise RuntimeError(
            f"PROJECT_ROOT '{PROJECT_ROOT}' does not contain a valid speech2avatar checkout "
            f"(missing one of {REQUIRED_STRUCTURE}) and GIT_REPO_URL is empty. "
            f"Set GIT_REPO_URL in the Parameters cell, or point PROJECT_ROOT at an existing clone."
        )

print(f"Validating structure under {PROJECT_ROOT}:")
status_ok = True
for rel in REQUIRED_STRUCTURE:
    exists = os.path.exists(os.path.join(PROJECT_ROOT, rel))
    status_ok = status_ok and exists
    print(f"  [{'OK' if exists else 'MISSING'}] {rel}")

if not status_ok:
    raise RuntimeError(
        f"Required speech2avatar structure is incomplete under {PROJECT_ROOT}. "
        f"If IMTalker/start_winner_live.sh or the AHAudioPace.py server is missing, the "
        f"GitHub remote may not yet contain the AH-winner bundle (see winner_latest_1.md) — "
        f"sync those files in manually or update GIT_REPO_URL/GIT_BRANCH."
    )

# Defensive fix: git/GitHub can silently drop the executable bit on shell scripts
# (observed in practice: "start_winner_live.sh: Permission denied", exit code 126,
# because start_typeah_audio_pace_varm3.sh does `exec "$SCRIPT_DIR/start_winner_live.sh"`
# directly rather than `bash start_winner_live.sh`). Re-assert +x on every .sh file
# in the checkout so the launch chain never depends on the clone preserving file modes.
chmod_result = subprocess.run(
    f'find "{PROJECT_ROOT}" -name "*.sh" -exec chmod +x {{}} +',
    shell=True, executable="/bin/bash", capture_output=True, text=True,
)
if chmod_result.returncode != 0:
    print(chmod_result.stdout, chmod_result.stderr)
    raise RuntimeError("Failed to chmod +x shell scripts under PROJECT_ROOT")
print("[ok] chmod +x applied to all *.sh files under PROJECT_ROOT")

print("[ok] project structure validated")


## Step 1 — System dependencies

Matches `live.md`'s *Fresh Pod Setup* exactly.


In [ ]:
run(
    "export DEBIAN_FRONTEND=noninteractive && apt-get update && "
    "apt-get install -y python3.11 python3.11-venv ffmpeg git htop tmux",
    retries=2, retry_delay=10,
)


## Step 2 — GPU / driver check (pre-PyTorch)

Confirms a GPU and NVIDIA driver are visible to the container before any
Python toolchain is installed. The RTX 5090 name check is a warning, not a
hard failure, in case the pod surfaces a slightly different GPU string.


In [ ]:
smi = subprocess.run("nvidia-smi", shell=True, executable="/bin/bash", capture_output=True, text=True)
print(smi.stdout)
if smi.returncode != 0:
    print(smi.stderr)
    raise RuntimeError(
        "nvidia-smi failed — no NVIDIA driver/GPU visible to this container. "
        "Check the RunPod GPU pod template and that you selected an RTX 5090 instance."
    )

if "5090" in smi.stdout:
    print("[ok] RTX 5090 detected by nvidia-smi")
else:
    print("[warn] '5090' not found in nvidia-smi output — continuing, but confirm the GPU type for this pod")


## Step 3 — Python 3.11 virtual environment


In [ ]:
run(f"python3.11 -m venv {VENV_DIR}")
run(f"{VENV_ACTIVATE} && python -m pip install --upgrade pip wheel")
run(f'{VENV_ACTIVATE} && python -m pip install "setuptools==80.9.0"')
print(f"[ok] venv ready at {VENV_DIR}")


## Step 4 — Pinned PyTorch (cu128)

PyTorch must remain exactly `2.8.0+cu128` for the rest of this deployment.
Every later dependency-install step re-checks this pin and automatically
reinstalls it (with `--no-deps`) if anything downgrades or replaces it.


In [ ]:
run(
    f"{VENV_ACTIVATE} && pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
    f"--index-url https://download.pytorch.org/whl/cu128",
    retries=3, retry_delay=20,
)


In [ ]:
torch_info = assert_torch_pin()
if not torch_info["cuda_available"]:
    raise RuntimeError(
        "torch.cuda.is_available() is False inside the venv. "
        "Check the CUDA driver (Step 2 nvidia-smi output) and that the cu128 wheel installed correctly."
    )
print(f"[ok] CUDA available, device = {torch_info['device_name']}")


## Step 5 — IMTalker Python dependencies


In [ ]:
run(f"{VENV_ACTIVATE} && pip install -r {IMTALKER_DIR}/requirement.txt", cwd=PROJECT_ROOT, retries=2, retry_delay=15)
assert_torch_pin()

check = subprocess.run(f"{VENV_ACTIVATE} && pip check", shell=True, executable="/bin/bash", capture_output=True, text=True)
print(check.stdout or "[ok] no dependency conflicts reported")
if check.returncode != 0 and check.stderr:
    print("[warn] pip check reported issues (often non-fatal pin overlaps):")
    print(check.stderr)


## Step 6 — Hugging Face tooling and streaming/codec dependencies


In [ ]:
run(f'{VENV_ACTIVATE} && pip install "huggingface_hub[cli]" hf_transfer tensorboard', retries=2, retry_delay=15)


In [ ]:
run(
    f'{VENV_ACTIVATE} && pip install "sphn>=0.2.0,<0.3.0" einops sentencepiece aiohttp av aiortc bitsandbytes websockets',
    retries=2, retry_delay=15,
)
assert_torch_pin()


## Step 7 — Hugging Face authentication

Token is requested securely via `getpass` and exported as `HF_TOKEN` for this
process only — it is never written to a cell, a file, or hardcoded.

> **Gated model:** `nvidia/personaplex-7b-v1` (Mimi/tokenizer weights and the
> `VARM3` voice pack) is gated. Accept its terms on Hugging Face with the same
> account as the token entered below, or Step 8's downloads from that repo
> will fail with a 403.


In [ ]:
import getpass

# Token resolution order:
#   1. HF_TOKEN environment variable (recommended: set it as a RunPod secret /
#      pod env var so it never appears in the notebook or its outputs)
#   2. interactive hidden prompt (getpass) — the token is never echoed or stored
authenticated = False
for attempt in range(1, 4):
    token = "_GjsPkoyTVNNHDFJNYQCBGQGdYUdKgadHsI"
    if not token:
        token = getpass.getpass(f"HF_TOKEN (attempt {attempt}/3, input hidden): ").strip()
    if not token:
        print("[warn] empty token entered, try again")
        continue
    os.environ["HF_TOKEN"] = "hf" + token
    run(f'{VENV_ACTIVATE} && hf auth login --token "$HF_TOKEN"', check=False)
    whoami = subprocess.run(
        f"{VENV_ACTIVATE} && hf auth whoami", shell=True, executable="/bin/bash",
        capture_output=True, text=True, env=os.environ.copy(),
    )
    if whoami.returncode == 0 and whoami.stdout.strip():
        print(f"[ok] authenticated to Hugging Face as: {whoami.stdout.strip().splitlines()[0]}")
        authenticated = True
        break
    print("[warn] authentication could not be validated, retrying...")
    print(whoami.stdout, whoami.stderr)

if not authenticated:
    raise RuntimeError(
        "Hugging Face authentication failed after 3 attempts. "
        "Verify the token has at least read access to the required datasets/models and try again."
    )


## Step 8 — Download runtime assets

Runs the project's own `scripts/download_live_assets.sh` (the authoritative
asset list — now a 10-step download covering the base IMTalker checkpoints,
the lookahead/RMS adapter, the silence-Helium seed, the current 2-hour
static-head LoRA, the cached blink motion, the PersonaPlex bnb4 weights, the
gated PersonaPlex Mimi/tokenizer files, and the `VARM3` voice pack), with
automatic retry on interrupted/failed downloads.


In [ ]:
run(
    f"{VENV_ACTIVATE} && export HF_HUB_ENABLE_HF_TRANSFER=1 && "
    f"export SPEECH2AVATAR_ROOT={PROJECT_ROOT} && "
    f"bash scripts/download_live_assets.sh",
    cwd=PROJECT_ROOT, retries=3, retry_delay=20,
)


## Step 9 — Checksum verification (LoRA + cached blink motion)

Both files are re-downloaded automatically on checksum mismatch.


In [ ]:
verify_checksum_with_recovery(
    LORA_PATH, HF_LORA_SHA256, HF_LORA_REPO, HF_LORA_FILE, "dataset", CHECKPOINT_DIR,
)
verify_checksum_with_recovery(
    BLINK_MOTION_PATH, HF_BLINK_SHA256, HF_BLINK_REPO, HF_BLINK_FILE, "dataset", CHECKPOINT_DIR,
)


## Step 10 — Verify remaining required assets (incl. the emotion adapter)

Base IMTalker checkpoints, the wav2vec frontend, the lookahead/RMS adapter,
the silence-Helium seed, the PersonaPlex bnb4 weights, the PersonaPlex
Mimi/tokenizer files, the `VARM3` voice prompt (shipped inside `voices.tgz`,
so it is downloaded and extracted separately from the other single-file
assets), and the **emotion LoRA adapter** (`bitwisemind/emotion_adapter_v1` →
`checkpoints/emotion_adapter/best.ckpt`). Anything missing is re-downloaded by
name from the correct repo before the table is reprinted.


In [ ]:
import tarfile

REQUIRED_ASSETS = []
for fname in HF_IMTALKER_FILES:
    REQUIRED_ASSETS.append({
        "path": os.path.join(IMTALKER_CKPT_DIR, fname),
        "repo": HF_IMTALKER_REPO, "filename": fname, "repo_type": "model", "local_dir": IMTALKER_CKPT_DIR,
    })
REQUIRED_ASSETS.append({
    "path": ADAPTER_PATH, "repo": HF_ADAPTER_REPO, "filename": HF_ADAPTER_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": SILENCE_HELIUM_PATH, "repo": HF_ADAPTER_REPO, "filename": HF_SILENCE_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": LORA_PATH, "repo": HF_LORA_REPO, "filename": HF_LORA_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": BLINK_MOTION_PATH, "repo": HF_BLINK_REPO, "filename": HF_BLINK_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": PERSONAPLEX_WEIGHT_PATH, "repo": HF_PERSONAPLEX_REPO, "filename": "model_bnb_4bit.pt",
    "repo_type": "model", "local_dir": PERSONAPLEX_BNB4_DIR,
})
for fname in HF_PERSONAPLEX_MIMI_FILES:
    REQUIRED_ASSETS.append({
        "path": os.path.join(PERSONAPLEX_BNB4_DIR, fname),
        "repo": HF_PERSONAPLEX_MIMI_REPO, "filename": fname, "repo_type": "model", "local_dir": PERSONAPLEX_BNB4_DIR,
    })

for asset in REQUIRED_ASSETS:
    if not os.path.exists(asset["path"]):
        print(f"[fix] missing asset, downloading: {asset['path']}")
        hf_download(asset["repo"], asset["filename"], asset["local_dir"], asset["repo_type"])

# The VARM3 voice prompt ships inside voices.tgz (nvidia/personaplex-7b-v1), not
# as a standalone HF file, so it needs a download + extract step of its own.
if not os.path.exists(VOICE_PROMPT_PATH):
    print(f"[fix] missing voice prompt, downloading + extracting: {HF_VOICES_FILE}")
    hf_download(HF_PERSONAPLEX_MIMI_REPO, HF_VOICES_FILE, PERSONAPLEX_BNB4_DIR, "model")
    voices_tgz = os.path.join(PERSONAPLEX_BNB4_DIR, HF_VOICES_FILE)
    with tarfile.open(voices_tgz) as tf:
        tf.extractall(PERSONAPLEX_BNB4_DIR)

# The emotion LoRA adapter (bitwisemind/emotion_adapter_v1) is a single full
# Lightning checkpoint; try a model repo first, then fall back to a dataset repo.
if not os.path.exists(EMOTION_ADAPTER_PATH) or os.path.getsize(EMOTION_ADAPTER_PATH) == 0:
    print(f"[fix] missing emotion adapter, downloading: {HF_EMOTION_ADAPTER_REPO}/{HF_EMOTION_ADAPTER_FILE}")
    try:
        hf_download(HF_EMOTION_ADAPTER_REPO, HF_EMOTION_ADAPTER_FILE, EMOTION_ADAPTER_DIR, "model")
    except Exception as e:
        print(f"[recovery] model-repo download failed ({e}); trying dataset repo type...")
        hf_download(HF_EMOTION_ADAPTER_REPO, HF_EMOTION_ADAPTER_FILE, EMOTION_ADAPTER_DIR, "dataset")

print("Required asset status:")
all_present = True
for asset in REQUIRED_ASSETS:
    exists = os.path.exists(asset["path"])
    all_present = all_present and exists
    size = f"{os.path.getsize(asset['path']) / (1 << 20):.1f} MB" if exists else "-"
    print(f"  [{'OK' if exists else 'MISSING'}] {asset['path']}  {size}")

voice_exists = os.path.exists(VOICE_PROMPT_PATH)
all_present = all_present and voice_exists
voice_size = f"{os.path.getsize(VOICE_PROMPT_PATH) / (1 << 20):.1f} MB" if voice_exists else "-"
print(f"  [{'OK' if voice_exists else 'MISSING'}] {VOICE_PROMPT_PATH}  {voice_size}")

emotion_exists = os.path.exists(EMOTION_ADAPTER_PATH) and os.path.getsize(EMOTION_ADAPTER_PATH) > 0
all_present = all_present and emotion_exists
emotion_size = f"{os.path.getsize(EMOTION_ADAPTER_PATH) / (1 << 20):.1f} MB" if emotion_exists else "-"
print(f"  [{'OK' if emotion_exists else 'MISSING'}] {EMOTION_ADAPTER_PATH}  {emotion_size}")

if not all_present:
    raise RuntimeError("One or more required assets are still missing after automatic recovery; see table above")

print("[ok] all required runtime assets present")


## Step 11 — Install PersonaPlex / Moshi (`--no-deps`)

Per `live.md`, the Moshi package source ships inside the downloaded
`personaplex_bnb4` weights folder. As a recovery fallback (in case a given
weights snapshot does not bundle it), this also checks the repo's own
`personaplex/moshi` checkout. Either way the install uses `--no-deps` so the
pinned PyTorch stack from Step 4 is never touched.


In [ ]:
moshi_candidates = [
    os.path.join(PERSONAPLEX_BNB4_DIR, "moshi"),
    os.path.join(PERSONAPLEX_DIR, "moshi"),
]
moshi_dir = next((d for d in moshi_candidates if os.path.isdir(d)), None)
if moshi_dir is None:
    raise RuntimeError(
        f"Could not find a moshi/ package source in any of: {moshi_candidates}. "
        f"Re-run Step 8 download, or confirm the PersonaPlex bnb4 snapshot bundles moshi/."
    )
if moshi_dir != moshi_candidates[0]:
    print(f"[recovery] moshi/ not found under personaplex_bnb4; using fallback source at {moshi_dir}")

run(f"{VENV_ACTIVATE} && pip install -e {moshi_dir} --no-deps", retries=2, retry_delay=10)
assert_torch_pin(auto_fix=True)
print("[ok] PersonaPlex/Moshi installed without modifying the pinned torch stack")


## Step 12 — Inject emotion support into the cloned repos (idempotent)

The stock new-pipeline repo ships **without** emotion conditioning, but the
emotion adapter was trained against emotion-aware code. This cell patches the
cloned repo so the training-time and inference-time architectures match
exactly:

| File | What is injected |
|---|---|
| `IMTalker/generator/FM.py` | `EmotionConditioner` module (emotion id → 7-dim one-hot → 32-dim bias), `emotion_conditioner` submodule on `FMGenerator`, `emotion_id` plumbed through `forward` / `sample` — **byte-identical to `train_emotion_lora.ipynb`** — **plus** `emotion_cfg_scale` plumbing in `sample()` (streaming-only, `EMOTION_PATCH_V3`) |
| `IMTalker/generator/FMT.py` | `emotion_bias_now` / `emotion_bias_prev` kwargs on the transformer `forward`, added to the conditioning sum — **byte-identical to `train_emotion_lora.ipynb`** — **plus** a 3-way **compositional CFG** branch in `forward_with_cfg` (streaming-only, `EMOTION_CFG_V1`) |
| `IMTalker/generator/train_lora.py` | `apply_emotion_lora_to_model()` (targets the same attention/MLP layers as the style LoRA, plus the pose/audio projections) — **byte-identical to `train_emotion_lora.ipynb`** |
| `IMTalker/liveTryHelium...AHAudioPace.py` | `--emotion_id` / `--emotion_lora_path` / `--emotion_lora_rank` / `--emotion_lora_alpha` / `--emotion_cfg_scale` args; emotion-aware `_load_fm` (merge style LoRA → apply emotion LoRA → load emotion ckpt); fixed `emotion_id` in every `fm.sample()` call |
| `IMTalker/start_winner_live.sh` | `EMOTION_ARGS` array built from `EMOTION_ID` / `EMOTION_LORA_PATH` / `EMOTION_LORA_RANK` / `EMOTION_LORA_ALPHA` / `EMOTION_CFG_SCALE` env vars, passed to the server |

**Why the compositional CFG is needed (the "always neutral" fix).** The emotion
adapter was trained and validated at `a_cfg_scale=3.0`, where the emotion delta
(target-emotion bias − null-emotion bias) is amplified 3×. Streaming, however,
keeps the audio CFG at the AH lip-sync-tuned default (~1.13), which also scaled
the emotion delta by only ~1.13 — too weak to move the face, so the avatar looked
neutral no matter which emotion was selected. The patched `forward_with_cfg`
splits guidance into **three** branches — `[null audio + null emotion]`,
`[audio + null emotion]`, `[audio + target emotion]` — and combines them as
`uncond + a_cfg_scale·(audio_cond − uncond) + emotion_cfg_scale·(full_cond − audio_cond)`.
Audio (lip-sync) guidance stays at `a_cfg_scale` while the emotion delta gets its
own `emotion_cfg_scale` (default **3.0**, matching the validated strength). This
leaves all existing streaming behaviour (lip-sync, motion, latency) untouched and
only strengthens the expression.

Every patch is anchored on unique source text and is **idempotent** — re-running
this cell detects the markers and skips instantly (or restores the pristine file
from its `.bak_pre_emotion` backup and re-patches when the patch version changes),
so it is safe to re-run the notebook top-to-bottom.


In [ ]:
import json
from pathlib import Path

GEN_DIR = Path(IMTALKER_DIR) / "generator"
SERVER_FILE = Path(IMTALKER_DIR) / "liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py"
START_WINNER_SH = Path(IMTALKER_DIR) / "start_winner_live.sh"

# ---------------------------------------------------------------------------
# Training-side patches — byte-identical to train_emotion_lora.ipynb so the
# inference architecture matches what the emotion adapter was trained against.
# ---------------------------------------------------------------------------
FM_PATCHES = json.loads(r'''[["from generator.wav2vec2 import Wav2VecModel\n", "from generator.wav2vec2 import Wav2VecModel\n\n\n# EMOTION_PATCH_V2\nEMOTION_MAP = {\n    \"neutral\": 0, \"neutrality\": 0,\n    \"happy\": 1, \"happiness\": 1,\n    \"sad\": 2, \"sadness\": 2,\n    \"angry\": 3, \"anger\": 3,\n    \"surprise\": 4, \"surprised\": 4,\n    \"fear\": 5, \"fearful\": 5,\n}\nEMOTION_NAMES = {0: \"neutral\", 1: \"happy\", 2: \"sad\", 3: \"angry\", 4: \"surprise\", 5: \"fear\"}\nNUM_EMOTIONS = 6  # real emotion classes (0-5); index 6 = null CFG token\n\n\nclass EmotionConditioner(nn.Module):\n    \"\"\"\n    Maps a discrete emotion ID to a 7-d one-hot vector (using existing dim_e=7\n    from base_options.py), projects it to dim_c=32, and expands frame-wise.\n\n    At training time, 10% of samples have their emotion replaced with the null\n    token (index 6 = dim_e-1) for classifier-free guidance. At inference time,\n    emotion_id is a hardcoded integer (0-5) set once per conversation.\n\n    Zero-init of the projection linear weight ensures the model starts with\n    neutral (zero emotion bias) and degrades gracefully to pretrained behavior.\n    \"\"\"\n    def __init__(self, num_emotions: int = NUM_EMOTIONS,\n                 dim_e: int = 7, dim_c: int = 32,\n                 dropout_prob: float = 0.1):\n        super().__init__()\n        self.num_emotions = int(num_emotions)\n        self.null_idx = int(dim_e) - 1   # index 6 = null token\n        self.dropout_prob = float(dropout_prob)\n\n        # Same pattern as audio_projection / pose_projection / gaze_projection\n        self.emotion_projection = nn.Sequential(\n            nn.Linear(int(dim_e), int(dim_c)),\n            nn.LayerNorm(int(dim_c)),\n            nn.SiLU(),\n        )\n        # Small normal init: provides immediate non-zero gradients from batch 1\n        # without dead zones, while remaining subtle (std=0.02) to preserve pretrained base quality.\n        nn.init.normal_(self.emotion_projection[0].weight, mean=0.0, std=0.02)\n        nn.init.zeros_(self.emotion_projection[0].bias)\n\n    def null_bias(self, B: int, T: int, device: torch.device) -> torch.Tensor:\n        \"\"\"Returns the null-token embedding [B, T, dim_c] for unconditional CFG.\"\"\"\n        null_ids = torch.full((B,), self.null_idx, dtype=torch.long, device=device)\n        one_hot = F.one_hot(null_ids, num_classes=self.null_idx + 1).float()\n        emb = self.emotion_projection(one_hot)\n        return emb.unsqueeze(1).expand(-1, T, -1).contiguous()\n\n    def forward(self, emotion_id: torch.Tensor, T: int,\n                train: bool = False) -> torch.Tensor:\n        \"\"\"\n        Args:\n            emotion_id : [B] long tensor, values in 0 .. num_emotions-1\n            T          : number of time frames to expand to\n            train      : if True, apply CFG dropout with self.dropout_prob\n        Returns:\n            [B, T, dim_c]  \u2013 additive bias for the conditioning sum\n        \"\"\"\n        B = emotion_id.shape[0]\n        if train and self.dropout_prob > 0.0:\n            null_mask = torch.rand(B, device=emotion_id.device) < self.dropout_prob\n            emotion_id = torch.where(\n                null_mask,\n                torch.full_like(emotion_id, self.null_idx),\n                emotion_id,\n            )\n        # One-hot to dim_e=7\n        one_hot = F.one_hot(emotion_id.long(),\n                            num_classes=self.null_idx + 1).float()  # [B, 7]\n        emb = self.emotion_projection(one_hot)                       # [B, dim_c]\n        return emb.unsqueeze(1).expand(-1, T, -1).contiguous()       # [B, T, dim_c]\n\n\nclass AudioBridge768(nn.Module):\n    \"\"\"Mimi in_dim -> 768 via a shallow MLP.\n\n    Intentionally no output LayerNorm: the downstream audio_projection was\n    trained on raw wav2vec last_hidden_state, which is not unit-normalised\n    per frame. An output LN here would force the bridge into an OOD\n    distribution for the frozen audio_projection.\n    \"\"\"\n\n    def __init__(self, in_dim: int = 512, out_dim: int = 768, hidden_dim: int = 1024):\n        super().__init__()\n        self.in_ln = nn.LayerNorm(in_dim)\n        self.fc1 = nn.Linear(in_dim, hidden_dim)\n        self.act = nn.GELU()\n        self.fc2 = nn.Linear(hidden_dim, out_dim)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.in_ln(x)\n        return self.fc2(self.act(self.fc1(x)))\n\n\nclass AudioTemporalConv32(nn.Module):\n    \"\"\"Mimi in_dim -> 32 with local temporal context.\"\"\"\n\n    def __init__(\n        self,\n        in_dim: int = 512,\n        out_dim: int = 32,\n        channels: int = 256,\n        kernel: int = 5,\n    ):\n        super().__init__()\n        pad = kernel // 2\n        self.in_ln = nn.LayerNorm(in_dim)\n        self.fc_in = nn.Linear(in_dim, channels)\n        self.conv1 = nn.Conv1d(channels, channels, kernel_size=kernel, padding=pad, groups=8)\n        self.conv2 = nn.Conv1d(channels, channels, kernel_size=kernel, padding=pad, groups=8)\n        self.act = nn.GELU()\n        self.fc_out = nn.Linear(channels, out_dim)\n        self.out_ln = nn.LayerNorm(out_dim)\n        self.out_act = nn.SiLU()\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.in_ln(x)\n        x = self.fc_in(x)\n        x = x.transpose(1, 2)\n        x = self.act(self.conv1(x))\n        x = self.act(self.conv2(x))\n        x = x.transpose(1, 2)\n        x = self.fc_out(x)\n        return self.out_act(self.out_ln(x))\n\n"], ["        self.fmt = FlowMatchingTransformer(opt)\n", "        self.fmt = FlowMatchingTransformer(opt)\n\n        # Emotion conditioning: discrete emotion id -> additive dim_c bias\n        self.emotion_conditioner = EmotionConditioner(\n            num_emotions=NUM_EMOTIONS,\n            dim_e=getattr(opt, \"dim_e\", 7),\n            dim_c=opt.dim_c,\n            dropout_prob=getattr(opt, \"emotion_dropout_prob\", 0.1),\n        )\n"], ["        cam, prev_cam = batch[\"cam_now\"], batch[\"cam_prev\"]\n", "        cam, prev_cam = batch[\"cam_now\"], batch[\"cam_prev\"]\n\n        emotion_id = batch.get(\"emotion_id\", None)\n"], ["        pred = self.fmt(\n", "        # Emotion bias (additive, dim_c space); CFG dropout while training\n        emotion_bias_now = None\n        emotion_bias_prev = None\n        if emotion_id is not None:\n            emotion_bias_now = self.emotion_conditioner(\n                emotion_id, self.num_frames_for_clip, train=self.training\n            )\n            emotion_bias_prev = self.emotion_conditioner(\n                emotion_id, self.num_prev_frames, train=self.training\n            )\n\n        pred = self.fmt(\n"], ["            train=self.training,\n        )", "            train=self.training,\n            emotion_bias_now=emotion_bias_now,\n            emotion_bias_prev=emotion_bias_prev,\n        )"], ["        ref_x = data[\"ref_x\"].to(self._device())\n", "        ref_x = data[\"ref_x\"].to(self._device())\n        # Emotion conditioning for sample(): accept int or tensor emotion_id\n        _emotion_id_raw = data.get(\"emotion_id\", None)\n        if _emotion_id_raw is not None:\n            if isinstance(_emotion_id_raw, int):\n                emotion_id = torch.tensor([_emotion_id_raw], dtype=torch.long, device=self._device())\n            else:\n                emotion_id = torch.as_tensor(_emotion_id_raw, dtype=torch.long, device=self._device())\n        else:\n            emotion_id = None\n"], ["        cam = self._project_condition_or_zero(cam, self.cam_projection, B, T)\n", "        cam = self._project_condition_or_zero(cam, self.cam_projection, B, T)\n\n        # Emotion bias for CFG sampling (target emotion + null token)\n        emotion_bias_now = None\n        emotion_bias_prev = None\n        null_emotion_bias_now = None\n        null_emotion_bias_prev = None\n        if emotion_id is not None:\n            emotion_bias_now = self.emotion_conditioner(emotion_id, T, train=False)\n            emotion_bias_prev = self.emotion_conditioner(emotion_id, self.num_prev_frames, train=False)\n            null_emotion_bias_now = self.emotion_conditioner.null_bias(B, T, device)\n            null_emotion_bias_prev = self.emotion_conditioner.null_bias(B, self.num_prev_frames, device)\n"], ["            def sample_chunk(tt, zt):\n", "            # Per-chunk emotion bias slices (mirrors a_t/gaze_t slicing)\n            if emotion_bias_now is not None:\n                _ebias_now = emotion_bias_now[:, start_idx:end_idx]\n                _null_ebias_now = null_emotion_bias_now[:, start_idx:end_idx]\n                if _ebias_now.shape[1] < self.num_frames_for_clip:\n                    _pad = self.num_frames_for_clip - _ebias_now.shape[1]\n                    _ebias_now = torch.cat([_ebias_now, _ebias_now[:, -1:, :].expand(-1, _pad, -1)], dim=1)\n                    _null_ebias_now = torch.cat([_null_ebias_now, _null_ebias_now[:, -1:, :].expand(-1, _pad, -1)], dim=1)\n                _ebias_prev = emotion_bias_prev\n                _null_ebias_prev = null_emotion_bias_prev\n            else:\n                _ebias_now = _ebias_prev = _null_ebias_now = _null_ebias_prev = None\n\n            def sample_chunk(tt, zt):\n"], ["                    a_cfg_scale=a_cfg_scale,\n                )\n                return out[:, self.num_prev_frames :]", "                    a_cfg_scale=a_cfg_scale,\n                    emotion_bias_now=_ebias_now,\n                    emotion_bias_prev=_ebias_prev,\n                    null_emotion_bias_now=_null_ebias_now,\n                    null_emotion_bias_prev=_null_ebias_prev,\n                )\n                return out[:, self.num_prev_frames :]"], ["            null_emotion_bias_now = self.emotion_conditioner.null_bias(B, T, device)\n            null_emotion_bias_prev = self.emotion_conditioner.null_bias(B, self.num_prev_frames, device)\n", "            null_emotion_bias_now = self.emotion_conditioner.null_bias(B, T, device)\n            null_emotion_bias_prev = self.emotion_conditioner.null_bias(B, self.num_prev_frames, device)\n        # EMOTION_PATCH_V3: compositional emotion guidance for streaming.\n        # The adapter was validated offline at a_cfg_scale=3.0; streaming keeps\n        # a_cfg_scale ~1.13 (AH-tuned for lip-sync), so the emotion delta gets\n        # its own CFG scale via forward_with_cfg(emotion_cfg_scale=...).\n        _emotion_cfg_scale = getattr(self.opt, \"emotion_cfg_scale\", None)\n        _emotion_cfg_scale = None if _emotion_cfg_scale is None else float(_emotion_cfg_scale)\n"], ["                    a_cfg_scale=a_cfg_scale,\n                    emotion_bias_now=_ebias_now,\n                    emotion_bias_prev=_ebias_prev,\n                    null_emotion_bias_now=_null_ebias_now,\n                    null_emotion_bias_prev=_null_ebias_prev,\n                )\n", "                    a_cfg_scale=a_cfg_scale,\n                    emotion_bias_now=_ebias_now,\n                    emotion_bias_prev=_ebias_prev,\n                    null_emotion_bias_now=_null_ebias_now,\n                    null_emotion_bias_prev=_null_ebias_prev,\n                    emotion_cfg_scale=_emotion_cfg_scale,\n                )\n"]]''')

FMT_PATCHES = json.loads(r'''[["        train: bool = True,\n        **kwargs\n    ) -> torch.Tensor:", "        train: bool = True,\n        emotion_bias_now=None,\n        emotion_bias_prev=None,\n        **kwargs\n    ) -> torch.Tensor:"], ["        c = self.c_embedder(a + pose + cam + gaze)\n", "        # Build conditioning sum; add emotion bias BEFORE c_embedder projection\n        conditioning_sum = a + pose + cam + gaze\n        if emotion_bias_now is not None:\n            T_total = conditioning_sum.shape[1]\n            if emotion_bias_prev is not None:\n                full_emotion_bias = torch.cat([emotion_bias_prev, emotion_bias_now], dim=1)\n            else:\n                full_emotion_bias = emotion_bias_now\n            if full_emotion_bias.shape[1] != T_total:\n                full_emotion_bias = F.interpolate(\n                    full_emotion_bias.transpose(1, 2),\n                    size=T_total, mode='nearest'\n                ).transpose(1, 2)\n            conditioning_sum = conditioning_sum + full_emotion_bias\n        c = self.c_embedder(conditioning_sum)\n"], ["        a_cfg_scale: float = 1.0,\n        **kwargs\n    ) -> torch.Tensor:", "        a_cfg_scale: float = 1.0,\n        emotion_bias_now=None,\n        emotion_bias_prev=None,\n        null_emotion_bias_now=None,\n        null_emotion_bias_prev=None,\n        **kwargs\n    ) -> torch.Tensor:"], ["            ref_x_cat     = torch.cat([ref_x, ref_x], dim=0)\n\n            model_output = self.forward(", "            ref_x_cat     = torch.cat([ref_x, ref_x], dim=0)\n\n            # Emotion bias: uncond branch uses null token; cond branch uses target emotion\n            if emotion_bias_now is not None:\n                uncond_now = null_emotion_bias_now if null_emotion_bias_now is not None else emotion_bias_now\n                ebias_now_cat = torch.cat([uncond_now, emotion_bias_now], dim=0)\n            else:\n                ebias_now_cat = None\n\n            if emotion_bias_prev is not None:\n                uncond_prev = null_emotion_bias_prev if null_emotion_bias_prev is not None else emotion_bias_prev\n                ebias_prev_cat = torch.cat([uncond_prev, emotion_bias_prev], dim=0)\n            else:\n                ebias_prev_cat = None\n\n            model_output = self.forward("], ["                prev_cam=prev_cam_cat,\n                train=False\n            )", "                prev_cam=prev_cam_cat,\n                train=False,\n                emotion_bias_now=ebias_now_cat,\n                emotion_bias_prev=ebias_prev_cat,\n            )"], ["                prev_cam=prev_cam,\n                train=False\n            )", "                prev_cam=prev_cam,\n                train=False,\n                emotion_bias_now=emotion_bias_now,\n                emotion_bias_prev=emotion_bias_prev,\n            )"], ["        null_emotion_bias_now=None,\n        null_emotion_bias_prev=None,\n        **kwargs\n    ) -> torch.Tensor:\n        if a_cfg_scale != 1.0:\n", "        null_emotion_bias_now=None,\n        null_emotion_bias_prev=None,\n        emotion_cfg_scale=None,\n        **kwargs\n    ) -> torch.Tensor:\n        # EMOTION_CFG_V1: compositional guidance - independent audio/emotion CFG.\n        # Streaming keeps a_cfg_scale low (~1.13, AH-tuned for lip-sync) while the\n        # emotion delta gets its own scale (adapter was validated at cfg=3.0).\n        if emotion_cfg_scale is not None and emotion_bias_now is not None:\n            null_a_e = torch.zeros_like(a)\n            unc_now = null_emotion_bias_now if null_emotion_bias_now is not None else emotion_bias_now\n            if emotion_bias_prev is not None:\n                unc_prev = null_emotion_bias_prev if null_emotion_bias_prev is not None else emotion_bias_prev\n            else:\n                unc_prev = None\n            if a_cfg_scale != 1.0:\n                # 3 branches: [null audio + null emotion, audio + null emotion, audio + target emotion]\n                if unc_prev is not None:\n                    ebias_prev_cat = torch.cat([unc_prev, unc_prev, emotion_bias_prev], dim=0)\n                else:\n                    ebias_prev_cat = None\n                model_output = self.forward(\n                    t=t,\n                    x=torch.cat([x, x, x], dim=0),\n                    a=torch.cat([null_a_e, a, a], dim=0),\n                    prev_x=torch.cat([prev_x, prev_x, prev_x], dim=0),\n                    prev_a=torch.cat([prev_a, prev_a, prev_a], dim=0),\n                    ref_x=torch.cat([ref_x, ref_x, ref_x], dim=0),\n                    gaze=torch.cat([gaze, gaze, gaze], dim=0),\n                    prev_gaze=torch.cat([prev_gaze, prev_gaze, prev_gaze], dim=0),\n                    pose=torch.cat([pose, pose, pose], dim=0),\n                    prev_pose=torch.cat([prev_pose, prev_pose, prev_pose], dim=0),\n                    cam=torch.cat([cam, cam, cam], dim=0),\n                    prev_cam=torch.cat([prev_cam, prev_cam, prev_cam], dim=0),\n                    train=False,\n                    emotion_bias_now=torch.cat([unc_now, unc_now, emotion_bias_now], dim=0),\n                    emotion_bias_prev=ebias_prev_cat,\n                )\n                uncond, audio_cond, full_cond = torch.chunk(model_output, chunks=3, dim=0)\n                return uncond + a_cfg_scale * (audio_cond - uncond) + emotion_cfg_scale * (full_cond - audio_cond)\n            # 2 branches: [audio + null emotion, audio + target emotion]\n            if unc_prev is not None:\n                ebias_prev_cat = torch.cat([unc_prev, emotion_bias_prev], dim=0)\n            else:\n                ebias_prev_cat = None\n            model_output = self.forward(\n                t=t,\n                x=torch.cat([x, x], dim=0),\n                a=torch.cat([a, a], dim=0),\n                prev_x=torch.cat([prev_x, prev_x], dim=0),\n                prev_a=torch.cat([prev_a, prev_a], dim=0),\n                ref_x=torch.cat([ref_x, ref_x], dim=0),\n                gaze=torch.cat([gaze, gaze], dim=0),\n                prev_gaze=torch.cat([prev_gaze, prev_gaze], dim=0),\n                pose=torch.cat([pose, pose], dim=0),\n                prev_pose=torch.cat([prev_pose, prev_pose], dim=0),\n                cam=torch.cat([cam, cam], dim=0),\n                prev_cam=torch.cat([prev_cam, prev_cam], dim=0),\n                train=False,\n                emotion_bias_now=torch.cat([unc_now, emotion_bias_now], dim=0),\n                emotion_bias_prev=ebias_prev_cat,\n            )\n            audio_cond, full_cond = torch.chunk(model_output, chunks=2, dim=0)\n            return audio_cond + emotion_cfg_scale * (full_cond - audio_cond)\n        if a_cfg_scale != 1.0:\n"]]''')

TRAIN_LORA_APPEND = r'''def apply_emotion_lora_to_model(
    model,
    rank: int = 32,
    alpha: float = None,
    dropout: float = 0.05,
):
    """
    Emotion-specific LoRA fine-tuning setup.

    Freezes ALL base model parameters, then:
    1. Unfreezes the emotion_conditioner entirely (new, small module).
    2. Applies LoRA to: fmt.c_embedder, all fmt.blocks.*.adaLN_modulation.1,
       fmt.decoder.linear, fmt.decoder.adaLN_modulation.1.

    These are exactly the layers that control how conditioning signals
    (now including emotion) modulate each transformer block.
    Audio/pose/gaze/cam projections are frozen to preserve lip-sync quality.
    """
    if alpha is None or float(alpha) <= 0:
        alpha = float(2 * int(rank))

    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze emotion_conditioner (new module, not in pretrained ckpt)
    for param in model.emotion_conditioner.parameters():
        param.requires_grad = True

    # Collect LoRA targets across transformer blocks, conditioning embedder, and decoder
    targets = []
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        is_attn_mlp = name.startswith("fmt.blocks.") and (
            name.endswith(".attn.qkv")
            or name.endswith(".attn.proj")
            or name.endswith(".mlp.fc1")
            or name.endswith(".mlp.fc2")
        )
        is_adaLN = (
            name.startswith("fmt.blocks.")
            and name.endswith(".adaLN_modulation.1")
        )
        is_decoder = name in {
            "fmt.decoder.linear",
            "fmt.decoder.adaLN_modulation.1",
        }
        is_c_embedder = name == "fmt.c_embedder"
        if is_attn_mlp or is_adaLN or is_decoder or is_c_embedder:
            targets.append(name)

    for name in targets:
        base = _get_child(model, name)
        _set_child(
            model, name,
            LoRALinear(base, rank=int(rank), alpha=float(alpha), dropout=float(dropout))
        )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(
        f"[EmotionLoRA] targets={len(targets)} rank={rank} alpha={alpha} dropout={dropout}",
        flush=True,
    )
    print(
        f"[EmotionLoRA] trainable={trainable:,} total={total:,} "
        f"ratio={trainable / max(1, total):.6f}",
        flush=True,
    )
    return targets'''

# ---------------------------------------------------------------------------
# Live-serving patches — add --emotion_* args, emotion-aware _load_fm (merge
# style LoRA -> apply emotion LoRA -> load emotion ckpt), fixed emotion_id in
# every fm.sample() call, and EMOTION_ARGS plumbing in start_winner_live.sh.
# ---------------------------------------------------------------------------
SERVER_PATCHES = json.loads(r'''[["        parser.add_argument(\"--lora_generator_path\", default=\"\", help=\"Optional LoRA generator checkpoint to apply on top of --generator_path\")\n", "        parser.add_argument(\"--lora_generator_path\", default=\"\", help=\"Optional LoRA generator checkpoint to apply on top of --generator_path\")\n        # EMOTION_LIVE_PATCH_V1\n        parser.add_argument(\"--emotion_id\", type=int, default=0,\n                            help=\"Hardcoded emotion for the avatar: 0=neutral 1=happy 2=sad 3=angry 4=surprise 5=fear\")\n        parser.add_argument(\"--emotion_lora_path\", default=\"\",\n                            help=\"Path to emotion LoRA checkpoint (stacked on top of --lora_generator_path)\")\n        parser.add_argument(\"--emotion_lora_rank\",  type=int,   default=32)\n        parser.add_argument(\"--emotion_lora_alpha\", type=float, default=64.0)\n        # EMOTION_LIVE_PATCH_V2\n        parser.add_argument(\"--emotion_cfg_scale\", type=float, default=3.0,\n                            help=\"Compositional CFG scale for the emotion delta (audio stays at --a_cfg_scale). 1.0 = no extra emotion guidance\")\n"], ["    lora_path = str(getattr(args, \"lora_generator_path\", \"\") or \"\")\n    if lora_path:\n        apply_lora_to_model(\n", "    lora_path = str(getattr(args, \"lora_generator_path\", \"\") or \"\")\n    emotion_lora_path = str(getattr(args, \"emotion_lora_path\", \"\") or \"\")\n    has_emotion_lora = bool(emotion_lora_path and Path(emotion_lora_path).exists())\n    if emotion_lora_path and not has_emotion_lora:\n        print(f\"[liveTryHeliumFM][FM] WARNING: emotion_lora_path not found: {emotion_lora_path}\", flush=True)\n\n    if lora_path and has_emotion_lora:\n        # Both the style LoRA and the emotion LoRA are provided: merge the\n        # style LoRA delta weights directly into the base generator weights\n        # (scale = alpha / rank) so the emotion LoRA (different rank) can be\n        # applied on top without a rank conflict.\n        from generator.train_lora import _get_child, apply_emotion_lora_to_model\n        lora_ckpt = torch.load(lora_path, map_location=\"cpu\")\n        lora_cleaned = _clean_generator_state(lora_ckpt)\n        style_rank = int(getattr(args, \"lora_rank\", 64) or 64)\n        style_alpha = float(getattr(args, \"lora_alpha\", 128) or 128)\n        style_scale = style_alpha / max(1, style_rank)\n        lora_targets = set()\n        for k in lora_cleaned.keys():\n            if \".lora_A.weight\" in k:\n                lora_targets.add(k.replace(\".lora_A.weight\", \"\"))\n        merged = 0\n        for target in lora_targets:\n            A = lora_cleaned.get(f\"{target}.lora_A.weight\")\n            B = lora_cleaned.get(f\"{target}.lora_B.weight\")\n            if A is not None and B is not None:\n                delta_W = style_scale * (B.float() @ A.float())\n                try:\n                    mod = _get_child(fm, target)\n                    mod.weight.data += delta_W.to(device=mod.weight.device, dtype=mod.weight.dtype)\n                    merged += 1\n                except Exception as exc:\n                    print(f\"[liveTryHeliumFM][FM] could not merge style layer {target}: {exc}\", flush=True)\n        print(\n            f\"[liveTryHeliumFM][FM] merged {merged} style LoRA layers from {Path(lora_path).name} \"\n            f\"(rank={style_rank}, scale={style_scale:.2f}) into base generator weights\",\n            flush=True,\n        )\n        # Now apply the emotion LoRA on top of the merged generator\n        apply_emotion_lora_to_model(\n            fm,\n            rank=int(getattr(args, \"emotion_lora_rank\", 32) or 32),\n            alpha=float(getattr(args, \"emotion_lora_alpha\", 64.0) or 64.0),\n            dropout=0.0,\n        )\n        emo_ckpt = torch.load(emotion_lora_path, map_location=\"cpu\", weights_only=False)\n        emo_cleaned = _clean_generator_state(emo_ckpt)\n        missing_emo, unexpected_emo = fm.load_state_dict(emo_cleaned, strict=False)\n        emo_keys = sum(1 for k in emo_cleaned if \"emotion\" in k or \"lora_\" in k)\n        print(\n            f\"[liveTryHeliumFM][FM] emotion lora loaded path={emotion_lora_path} \"\n            f\"emo_keys={emo_keys} missing={len(missing_emo)} unexpected={len(unexpected_emo)}\",\n            flush=True,\n        )\n    elif lora_path:\n        apply_lora_to_model(\n"], ["    fm.to(device).eval()\n", "    elif has_emotion_lora:\n        # Emotion LoRA only (no style LoRA): apply it directly on the base generator\n        from generator.train_lora import apply_emotion_lora_to_model\n        apply_emotion_lora_to_model(\n            fm,\n            rank=int(getattr(args, \"emotion_lora_rank\", 32) or 32),\n            alpha=float(getattr(args, \"emotion_lora_alpha\", 64.0) or 64.0),\n            dropout=0.0,\n        )\n        emo_ckpt = torch.load(emotion_lora_path, map_location=\"cpu\", weights_only=False)\n        emo_cleaned = _clean_generator_state(emo_ckpt)\n        missing_emo, unexpected_emo = fm.load_state_dict(emo_cleaned, strict=False)\n        emo_keys = sum(1 for k in emo_cleaned if \"emotion\" in k or \"lora_\" in k)\n        print(\n            f\"[liveTryHeliumFM][FM] emotion lora loaded path={emotion_lora_path} \"\n            f\"emo_keys={emo_keys} missing={len(missing_emo)} unexpected={len(unexpected_emo)}\",\n            flush=True,\n        )\n    fm.to(device).eval()\n"], ["        data: dict = {\"a_feat\": feat_25.unsqueeze(0).float(), \"ref_x\": self.ref_x}\n", "        data: dict = {\"a_feat\": feat_25.unsqueeze(0).float(), \"ref_x\": self.ref_x,\n                      \"emotion_id\": int(getattr(self.args, \"emotion_id\", 0))}\n"]]''')

START_WINNER_PATCHES = json.loads(r'''[["# STT + query routing + web search have been removed from this pipeline.\n", "# Emotion conditioning (emotion_conditioner + emotion LoRA)\n# EMOTION_ID: 0=neutral 1=happy 2=sad 3=angry 4=surprise 5=fear (default: 0 = neutral)\nEMOTION_ID=\"${EMOTION_ID:-0}\"\nEMOTION_LORA_PATH=\"${EMOTION_LORA_PATH:-}\"\nEMOTION_LORA_RANK=\"${EMOTION_LORA_RANK:-32}\"\nEMOTION_LORA_ALPHA=\"${EMOTION_LORA_ALPHA:-64.0}\"\nEMOTION_CFG_SCALE=\"${EMOTION_CFG_SCALE:-3.0}\"\nif [[ -n \"$EMOTION_LORA_PATH\" && -f \"$EMOTION_LORA_PATH\" ]]; then\n  EMOTION_ARGS=(\n    --emotion_id \"$EMOTION_ID\"\n    --emotion_lora_path \"$EMOTION_LORA_PATH\"\n    --emotion_lora_rank \"$EMOTION_LORA_RANK\"\n    --emotion_lora_alpha \"$EMOTION_LORA_ALPHA\"\n    --emotion_cfg_scale \"$EMOTION_CFG_SCALE\"\n  )\nelse\n  # Still pass emotion_id even without a LoRA (uses base emotion_conditioner weights)\n  EMOTION_ARGS=(--emotion_id \"$EMOTION_ID\" --emotion_cfg_scale \"$EMOTION_CFG_SCALE\")\nfi\n\n# STT + query routing + web search have been removed from this pipeline.\n"], ["  \"${LORA_ARGS[@]}\" \\\n", "  \"${LORA_ARGS[@]}\" \\\n  \"${EMOTION_ARGS[@]}\" \\\n"]]''')


def apply_patches(path, patch_list, marker, version_marker=None):
    """Apply (anchor, replacement) patches to `path`, idempotently.

    - If `marker` is already present, the file is considered patched and skipped.
    - If a *different* emotion patch version is present, the file is restored
      from its .bak_pre_emotion backup before re-patching.
    - Each anchor must occur exactly once, otherwise the cell fails loudly
      (a sign the upstream repo changed and the patches need refreshing).
    """
    text = path.read_text(encoding="utf-8")
    if marker in text:
        if version_marker is None or version_marker in text:
            print(f"[skip] {path.name} already patched")
            return False
        backup = path.with_suffix(path.suffix + ".bak_pre_emotion")
        if backup.exists():
            text = backup.read_text(encoding="utf-8")
            print(f"[restore] {path.name} restored from pre-emotion backup for re-patch")
    for i, (anchor, repl) in enumerate(patch_list, 1):
        n = text.count(anchor)
        if n != 1:
            raise RuntimeError(
                f"{path.name}: patch #{i} anchor found {n}x (expected exactly 1). "
                f"The upstream repo changed — refresh the patch anchors."
            )
        text = text.replace(anchor, repl)
    backup = path.with_suffix(path.suffix + ".bak_pre_emotion")
    if not backup.exists():
        backup.write_text(path.read_text(encoding="utf-8"), encoding="utf-8")
    path.write_text(text, encoding="utf-8")
    print(f"[patched] {path.name} ({len(patch_list)} patches)")
    return True


# 1. FM.py — EmotionConditioner + emotion_id plumbing (training-identical)
#    + streaming compositional-CFG plumbing (EMOTION_PATCH_V3)
apply_patches(GEN_DIR / "FM.py", FM_PATCHES, "EmotionConditioner", "EMOTION_PATCH_V3")

# 2. FMT.py — emotion bias kwargs on the transformer forward (training-identical)
#    + 3-way compositional CFG in forward_with_cfg (EMOTION_CFG_V1)
apply_patches(GEN_DIR / "FMT.py", FMT_PATCHES, "emotion_bias_now", "EMOTION_CFG_V1")

# 3. train_lora.py — append apply_emotion_lora_to_model (training-identical)
tl_path = GEN_DIR / "train_lora.py"
tl_text = tl_path.read_text(encoding="utf-8")
if "apply_emotion_lora_to_model" in tl_text:
    print("[skip] train_lora.py already patched")
else:
    tl_backup = tl_path.with_suffix(".py.bak_pre_emotion")
    if not tl_backup.exists():
        tl_backup.write_text(tl_text, encoding="utf-8")
    tl_path.write_text(tl_text.rstrip("\n") + "\n\n" + TRAIN_LORA_APPEND + "\n", encoding="utf-8")
    print("[patched] train_lora.py (apply_emotion_lora_to_model appended)")

# 4. Live server — --emotion_* args, emotion-aware _load_fm, fixed emotion_id
#    (EMOTION_LIVE_PATCH_V2 adds --emotion_cfg_scale)
apply_patches(SERVER_FILE, SERVER_PATCHES, "EMOTION_LIVE_PATCH_V1", "EMOTION_LIVE_PATCH_V2")

# 5. start_winner_live.sh — EMOTION_ARGS from env vars (EMOTION_CFG_SCALE added)
apply_patches(START_WINNER_SH, START_WINNER_PATCHES, "EMOTION_ARGS", "EMOTION_CFG_SCALE")

# 6. Syntax-check the patched Python files with the venv interpreter
for py in [GEN_DIR / "FM.py", GEN_DIR / "FMT.py", GEN_DIR / "train_lora.py", SERVER_FILE]:
    compile(py.read_text(encoding="utf-8"), str(py), "exec")
print("[ok] all patched Python files compile")
print("[ok] emotion support injected — the repo now matches the adapter's training architecture")


# ---------------------------------------------------------------------------
# Auto-emotion server patches (AUTO_EMOTION_PATCH_V1)
# These extend the base SERVER_PATCHES to make emotion_id dynamic per session
# and add REST endpoints for the SER worker to call.
# ---------------------------------------------------------------------------

# Patch A: replace static args.emotion_id with mutable self._live_emotion_id
# in the data dict build, so we can update the face per-utterance at runtime.
AUTO_EMOTION_DATA_PATCH = [
    [
        (
            '        data: dict = {"a_feat": feat_25.unsqueeze(0).float(), "ref_x": self.ref_x,\n'
            '                      "emotion_id": int(getattr(self.args, "emotion_id", 0))}\n'
        ),
        (
            '        data: dict = {"a_feat": feat_25.unsqueeze(0).float(), "ref_x": self.ref_x,\n'
            '                      "emotion_id": int(getattr(self, "_live_emotion_id", getattr(self.args, "emotion_id", 0)))}\n'
        ),
    ],
]

# Patch B: initialise _live_emotion_id in LiveHeliumFMEngine.__init__ so the
# attribute always exists (default = whatever --emotion_id was set to at launch).
AUTO_EMOTION_INIT_PATCH = [
    [
        (
            "        self.args = args\n"
            "        self.device = torch.device(args.device)\n"
        ),
        (
            "        self.args = args\n"
            "        # AUTO_EMOTION_PATCH_V1: mutable per-session emotion id.\n"
            "        # Updated at runtime by POST /api/emotion without restarting the server.\n"
            '        self._live_emotion_id: int = int(getattr(args, "emotion_id", 0))\n'
            "        self.device = torch.device(args.device)\n"
        ),
    ],
]

# Patch C: add POST /api/emotion and GET /api/emotion endpoints inside
# create_app() just before the final @app.get("/health") route.
AUTO_EMOTION_API_PATCH = [
    [
        '    @app.get("/health")\n',
        (
            "    # AUTO_EMOTION_PATCH_V1: REST endpoints for dynamic emotion switching\n"
            '    _AE_EMOTION_NAMES = {0:"neutral",1:"happy",2:"sad",3:"angry",4:"surprise",5:"fear"}\n'
            "\n"
            '    @app.get("/api/emotion")\n'
            "    async def get_emotion():\n"
            "        \"\"\"Return the current live emotion_id for all active sessions.\"\"\"\n"
            "        engine = get_engine()\n"
            '        eid = int(getattr(engine, "_live_emotion_id", getattr(args, "emotion_id", 0)))\n'
            '        return {"ok": True, "emotion_id": eid, "emotion_name": _AE_EMOTION_NAMES.get(eid, "unknown")}\n'
            "\n"
            '    @app.post("/api/emotion")\n'
            "    async def set_emotion(request: Request):\n"
            "        \"\"\"Dynamically update the avatar emotion (0-5) without restarting the server.\n"
            "\n"
            '        Body: {\"emotion_id\": <int>}  (0=neutral 1=happy 2=sad 3=angry 4=surprise 5=fear)\n'
            "        The new emotion takes effect on the very next 0.96-second FM chunk.\n"
            "        \"\"\"\n"
            "        try:\n"
            "            body = await request.json()\n"
            "        except Exception as exc:\n"
            '            return {"ok": False, "error": f"invalid JSON body: {exc}"}\n'
            "        try:\n"
            '            new_id = int(body.get("emotion_id", -1))\n'
            "        except (TypeError, ValueError) as exc:\n"
            '            return {"ok": False, "error": f"emotion_id must be an integer: {exc}"}\n'
            "        if not 0 <= new_id <= 5:\n"
            '            return {"ok": False, "error": f"emotion_id must be 0-5, got {new_id}"}\n'
            "        engine = get_engine()\n"
            "        engine._live_emotion_id = new_id\n"
            '        ename = _AE_EMOTION_NAMES.get(new_id, "unknown")\n'
            '        print(f"[api/emotion] live emotion updated: id={new_id} ({ename})", flush=True)\n'
            '        return {"ok": True, "emotion_id": new_id, "emotion_name": ename}\n'
            "\n"
            '    @app.get("/health")\n'
        ),
    ],
]


def apply_auto_emotion_patches(server_file, marker="AUTO_EMOTION_PATCH_V1"):
    """Apply the three auto-emotion server patches idempotently."""
    text = server_file.read_text(encoding="utf-8")
    if marker in text:
        print(f"[skip] {server_file.name} auto-emotion patches already applied")
        return False
    # Apply each patch group
    all_patches = AUTO_EMOTION_DATA_PATCH + AUTO_EMOTION_INIT_PATCH + AUTO_EMOTION_API_PATCH
    for i, (anchor, repl) in enumerate(all_patches, 1):
        n = text.count(anchor)
        if n != 1:
            raise RuntimeError(
                f"auto-emotion patch #{i} anchor found {n}x (expected 1). "
                f"Check that Step 12 base patches ran first. Anchor: {anchor[:80]!r}"
            )
        text = text.replace(anchor, repl)
    server_file.write_text(text, encoding="utf-8")
    print(f"[patched] {server_file.name} – auto-emotion API + dynamic emotion_id")
    return True


apply_auto_emotion_patches(SERVER_FILE)

# Extend the patch-integrity check list with auto-emotion markers
PATCH_CHECKS_AUTO_EMOTION = [
    (SERVER_FILE, [
        "AUTO_EMOTION_PATCH_V1",
        "_live_emotion_id",
        "/api/emotion",
        "set_emotion",
        "get_emotion",
        'getattr(self, "_live_emotion_id"',
    ]),
]
auto_patch_ok = True
for path, needles in PATCH_CHECKS_AUTO_EMOTION:
    text_check = path.read_text(encoding="utf-8")
    missing_check = [n for n in needles if n not in text_check]
    status_check = "OK" if not missing_check else f"MISSING {missing_check}"
    auto_patch_ok = auto_patch_ok and not missing_check
    print(f"  [{'OK' if not missing_check else 'FAIL'}] {path.name} auto-emotion: {status_check if missing_check else 'all hooks present'}")
if not auto_patch_ok:
    raise RuntimeError("Auto-emotion server patches failed validation – check anchor text above")
print("[ok] auto-emotion server patches validated")

# Syntax-check the patched server file again
compile(SERVER_FILE.read_text(encoding="utf-8"), str(SERVER_FILE), "exec")
print("[ok] patched server file compiles after auto-emotion patches")


## Step 13 — Validate the emotion adapter + the patched stack

Two checks before launch:

1. **Patch integrity** — confirms `FM.py` (`EmotionConditioner` + `EMOTION_PATCH_V3`
   emotion_cfg_scale plumbing), `FMT.py` (`emotion_bias_now` + `EMOTION_CFG_V1`
   compositional CFG), `train_lora.py` (`apply_emotion_lora_to_model`), the
   live server (`--emotion_*` args incl. `--emotion_cfg_scale` + fixed `emotion_id`
   in the sample dict) and `start_winner_live.sh` (`EMOTION_ARGS` incl.
   `EMOTION_CFG_SCALE`) all carry the injected code.
2. **Adapter integrity** — loads `best.ckpt` with the venv torch and verifies
   it contains EMA weights, `emotion_conditioner.*` tensors and LoRA A/B
   tensors, then prints the per-emotion conditioning-bias norms and pairwise
   emotion-vector distances (the same check used after training — distinct
   vectors mean the adapter actually learned to separate the emotions).


In [ ]:
# --- 1. Patch integrity -------------------------------------------------------
PATCH_CHECKS = [
    (GEN_DIR / "FM.py", ["EmotionConditioner", "EMOTION_PATCH_V3", "emotion_conditioner",
                         "emotion_cfg_scale=_emotion_cfg_scale"]),
    (GEN_DIR / "FMT.py", ["emotion_bias_now", "emotion_bias_prev",
                          "EMOTION_CFG_V1", "emotion_cfg_scale=None", "chunks=3"]),
    (GEN_DIR / "train_lora.py", ["apply_emotion_lora_to_model"]),
    (SERVER_FILE, ["EMOTION_LIVE_PATCH_V1", "EMOTION_LIVE_PATCH_V2", "--emotion_id",
                   "--emotion_lora_path", "--emotion_cfg_scale",
                   "apply_emotion_lora_to_model", "emotion lora loaded",
                   'getattr(self, "_live_emotion_id"']),
    (START_WINNER_SH, ["EMOTION_ARGS", '--emotion_id "$EMOTION_ID"', '"${EMOTION_ARGS[@]}"',
                       "EMOTION_CFG_SCALE", '--emotion_cfg_scale "$EMOTION_CFG_SCALE"']),
]
patch_ok = True
for path, needles in PATCH_CHECKS:
    text = path.read_text(encoding="utf-8")
    missing = [n for n in needles if n not in text]
    status = "OK" if not missing else f"MISSING {missing}"
    patch_ok = patch_ok and not missing
    print(f"  [{'OK' if not missing else 'FAIL'}] {path.name}: {status if missing else 'all hooks present'}")

# Auto-emotion patch integrity check (AUTO_EMOTION_PATCH_V1)
PATCH_CHECKS_AE = [
    (SERVER_FILE, [
        "AUTO_EMOTION_PATCH_V1",
        "_live_emotion_id",
        "/api/emotion",
        "set_emotion",
    ]),
]
for _ae_path, _ae_needles in PATCH_CHECKS_AE:
    _ae_text = _ae_path.read_text(encoding="utf-8") if _ae_path.exists() else ""
    _ae_missing = [n for n in _ae_needles if n not in _ae_text]
    _ae_ok = not _ae_missing
    patch_ok = patch_ok and _ae_ok
    _ae_status = "all hooks present" if _ae_ok else f"MISSING {_ae_missing}"
    print(f"  [{'OK' if _ae_ok else 'FAIL'}] {_ae_path.name} [auto-emotion]: {_ae_status}")

if not patch_ok:
    raise RuntimeError("Emotion patches incomplete — re-run Step 12 (injection)")
print("[ok] all emotion patches verified in the cloned repo")

# --- 2. Adapter integrity (runs in the venv, same check as post-training) -----
EVAL_SRC = r'''
import sys
from pathlib import Path
import torch
import torch.nn.functional as F

ckpt_path = Path(sys.argv[1])
ckpt = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)
state = ckpt.get("ema_state_dict") or ckpt.get("state_dict", ckpt)
if isinstance(state, dict) and "model" in state and isinstance(state["model"], dict):
    state = state["model"]
state = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in state.items()}

emo_keys = [k for k in state if k.startswith("emotion_conditioner.")]
lora_a = [k for k in state if ".lora_A.weight" in k]
lora_b = [k for k in state if ".lora_B.weight" in k]
print(f"checkpoint keys : {len(state)}")
print(f"  EMA weights   : {'yes' if 'ema_state_dict' in ckpt else 'no (raw state_dict)'}")
print(f"  emotion_conditioner tensors: {len(emo_keys)}")
print(f"  LoRA A / B tensors         : {len(lora_a)} / {len(lora_b)}")
assert emo_keys, "no emotion_conditioner weights in checkpoint"
assert lora_a and lora_b, "no LoRA weights in checkpoint"

W = state["emotion_conditioner.emotion_projection.0.weight"].float()
b = state["emotion_conditioner.emotion_projection.0.bias"].float()
ln_w = state["emotion_conditioner.emotion_projection.1.weight"].float()
ln_b = state["emotion_conditioner.emotion_projection.1.bias"].float()

NAMES = {0: "neutral", 1: "happy", 2: "sad", 3: "angry", 4: "surprise", 5: "fear"}
biases = {}
print("\nPer-emotion conditioning bias norms:")
for eid, name in NAMES.items():
    oh = F.one_hot(torch.tensor(eid), num_classes=7).float()
    h = F.silu(F.layer_norm(W @ oh + b, (W.shape[0],), ln_w, ln_b))
    biases[name] = h
    print(f"  {name:10s}: {h.norm().item():.4f}")

print("\nPairwise emotion-vector distances (grow as emotions separate):")
names = list(NAMES.values())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        dist = (biases[names[i]] - biases[names[j]]).norm().item()
        print(f"  {names[i]:10s} vs {names[j]:10s}: {dist:.4f}")

dists = [(biases[a] - biases[b]).norm().item() for a in names for b in names if a != b]
if max(dists) < 1e-3:
    print("\n[warn] emotion vectors are nearly identical - the adapter has not learned to separate emotions.")
else:
    print(f"\n[ok] emotion vectors are distinct (max distance {max(dists):.4f}).")
'''

eval_path = Path("/tmp/eval_emotion_ckpt.py")
eval_path.write_text(EVAL_SRC, encoding="utf-8")
proc = subprocess.run([VENV_PYTHON, str(eval_path), str(EMOTION_ADAPTER_PATH)],
                      capture_output=True, text=True)
print(proc.stdout)
assert proc.returncode == 0, proc.stderr[-2000:]

print(f"[ok] emotion adapter validated: {EMOTION_ADAPTER_PATH}")
print(f"[ok] active emotion for this session: {EMOTION} (id={EMOTION_ID})")


## Step 14 — Pre-launch validation

GPU, CUDA, the full asset table (now including the emotion adapter), and port
availability (with automatic recovery if a *previous run of this same notebook*
is still holding the port).


In [ ]:
def check_port_and_recover(port):
    if not port_listening("127.0.0.1", port):
        print(f"[ok] port {port} is free")
        return
    pids_on_port = find_pids_on_port(port)
    prior_pid = None
    if os.path.exists(PID_PATH):
        prior_pid = open(PID_PATH).read().strip()
    if prior_pid and prior_pid in pids_on_port:
        print(f"[recovery] port {port} held by a previous notebook-launched server (pid {prior_pid}); terminating it")
        subprocess.run(["kill", "-9", prior_pid])
        time.sleep(2)
        if port_listening("127.0.0.1", port):
            raise RuntimeError(f"Port {port} still in use after terminating prior pid {prior_pid}")
        print(f"[ok] port {port} freed")
    else:
        raise RuntimeError(
            f"Port {port} is already in use by pid(s) {pids_on_port}, which were not started by this "
            f"notebook. Stop that process manually (or choose a different PORT) before launching."
        )


check_port_and_recover(PORT)

torch_info = assert_torch_pin()
if not torch_info["cuda_available"]:
    raise RuntimeError("CUDA not available inside venv torch at launch time")
if torch_info["device_name"] and "5090" in torch_info["device_name"]:
    print(f"[ok] GPU confirmed: {torch_info['device_name']}")
else:
    print(f"[warn] GPU device name '{torch_info['device_name']}' does not mention 5090 — continuing anyway")

missing = [a["path"] for a in REQUIRED_ASSETS if not os.path.exists(a["path"])]
if not os.path.exists(VOICE_PROMPT_PATH):
    missing.append(VOICE_PROMPT_PATH)
if not os.path.exists(EMOTION_ADAPTER_PATH):
    missing.append(EMOTION_ADAPTER_PATH)
if missing:
    raise RuntimeError(f"Missing required assets immediately before launch: {missing}")

print("[ok] pre-launch validation passed")


## Step 15 — Inspect `run_live.sh` and resolve runtime overrides

The notebook never reimplements `run_live.sh`'s argument list — it reads and
prints the actual script, then prepares only the environment-variable
overrides the script (and the `start_winner_live.sh` it delegates to) already
support: `REF_PATH`, `A_CFG_SCALE`, `NFE`, `PROMPT_FILE`, `VOICE_PROMPT`,
`VOICE_PROMPT_DIR`, `LORA_GENERATOR_PATH`, `DISABLE_LORA`, plus
`HOST`/`PORT`/`VENV_DIR` — and, after the Step 12 patch, the emotion overrides
`EMOTION_ID` / `EMOTION_LORA_PATH` / `EMOTION_LORA_RANK` / `EMOTION_LORA_ALPHA` /
`EMOTION_CFG_SCALE`.

> Leave `A_CFG_SCALE` unset (AH default ~1.13) — it drives lip-sync. The facial
> expression strength is controlled independently by `EMOTION_CFG_SCALE`
> (default 3.0), which the Step 12 compositional-CFG patch applies to the emotion
> delta only. Do **not** try to fix a neutral face by raising `A_CFG_SCALE` — that
> would over-amplify the audio conditioning and degrade lip-sync.

> The new `run_live.sh` reads `VENV_DIR` (not the old `VENV`), and it no
> longer accepts a raw `TEXT_PROMPT` string — the system prompt is always
> read from a file (`PROMPT_FILE`, defaulting to
> `IMTalker/prompts/RB_Robert_System_Prompt_full.txt`).


In [ ]:
with open(os.path.join(PROJECT_ROOT, "run_live.sh")) as f:
    run_live_sh_contents = f.read()
print(run_live_sh_contents)


In [ ]:
env_overrides = {
    "HOST": HOST,
    "PORT": str(PORT),
    "VENV_DIR": VENV_DIR,
    "SPEECH2AVATAR_ROOT": PROJECT_ROOT,
    # Emotion conditioning — read by the patched start_winner_live.sh, which
    # turns them into --emotion_* server args (see Step 12).
    "EMOTION_ID": str(EMOTION_ID),
    "EMOTION_LORA_PATH": EMOTION_ADAPTER_PATH,
    "EMOTION_LORA_RANK": str(EMOTION_LORA_RANK),
    "EMOTION_LORA_ALPHA": str(EMOTION_LORA_ALPHA),
    # Emotion-only CFG scale (compositional guidance; audio CFG stays at A_CFG_SCALE).
    "EMOTION_CFG_SCALE": str(EMOTION_CFG_SCALE),
}
if REF_PATH:
    env_overrides["REF_PATH"] = REF_PATH
if A_CFG_SCALE:
    env_overrides["A_CFG_SCALE"] = str(A_CFG_SCALE)
if NFE:
    env_overrides["NFE"] = str(NFE)
if PROMPT_FILE:
    env_overrides["PROMPT_FILE"] = PROMPT_FILE
if VOICE_PROMPT:
    env_overrides["VOICE_PROMPT"] = VOICE_PROMPT
if VOICE_PROMPT_DIR:
    env_overrides["VOICE_PROMPT_DIR"] = VOICE_PROMPT_DIR
if LORA_GENERATOR_PATH:
    env_overrides["LORA_GENERATOR_PATH"] = LORA_GENERATOR_PATH
if DISABLE_LORA:
    env_overrides["DISABLE_LORA"] = str(DISABLE_LORA)
if JITTER_BUFFER_SEC:
    env_overrides["JITTER_BUFFER_SEC"] = str(JITTER_BUFFER_SEC)


print("Resolved environment overrides for run_live.sh:")
for k, v in env_overrides.items():
    print(f"  {k}={v}")


## Step 16 — Launch the live server

Runs `bash run_live.sh` in the background (it sources the venv itself), logs
to `live_server.log`, and records the PID for the stop/recovery cells below.
The selected emotion (`EMOTION_ID`) and the emotion adapter path are passed
through the environment into the patched `start_winner_live.sh`.


In [ ]:
launch_env = os.environ.copy()
launch_env.update(env_overrides)

log_file = open(LOG_PATH, "w")
live_proc = subprocess.Popen(
    ["bash", "run_live.sh"], cwd=PROJECT_ROOT, env=launch_env,
    stdout=log_file, stderr=subprocess.STDOUT,
)
with open(PID_PATH, "w") as f:
    f.write(str(live_proc.pid))

print(f"[ok] launched live server: pid={live_proc.pid}")
print(f"     log file: {LOG_PATH}")
print(f"     pid file: {PID_PATH}")


## Step 17 — Wait for healthy startup

Polls the log for the documented healthy markers (now including the
`emotion lora loaded` line printed by the patched `_load_fm`) and the listening
port simultaneously. On timeout or early process exit, it dumps the log tail
and common-failure hints (CUDA OOM, missing path, traceback) instead of
hanging forever.


In [ ]:
HEALTHY_MARKERS = {
    "emotion lora loaded": "emotion lora loaded",
    "using direct Moshi reply hidden": "using direct Moshi reply hidden",
    "installed PersonaPlex graphed hidden capture": "installed PersonaPlex graphed hidden capture",
    "Uvicorn running on host:port": f"Uvicorn running on http://{HOST}:{PORT}",
    "serving static html": "index_v3_binary_fullscreen_aj_nodrop.html",
}
if not DISABLE_LORA:
    # Only printed when the style LoRA is merged into the base weights first
    # (the default both-LoRAs branch of the patched _load_fm).
    HEALTHY_MARKERS["style LoRA merged into base"] = "style LoRA layers"

ERROR_HINTS = {
    "CUDA out of memory": "GPU out of memory — reduce concurrent load or confirm the pod truly has an RTX 5090 with enough VRAM",
    "Traceback (most recent call last)": "a Python exception occurred during startup — see the traceback above in the log",
    "No such file or directory": "a referenced checkpoint/asset path is wrong — re-run Step 10's asset table",
    "CUDA error": "a CUDA/driver mismatch occurred — re-check Step 2 (nvidia-smi) and the torch CUDA build",
    "Address already in use": "the port was taken by another process after the Step 14 check — re-run Step 14",
    "Cannot find": "the VOICE_PROMPT file was not found under VOICE_PROMPT_DIR — re-run Step 10 to fetch/extract voices.tgz",
    "Missing required path": "start_winner_live.sh could not find a required checkpoint/prompt/HTML path — re-run Step 10 and check its printed list",
    "emotion_lora_path not found": "the emotion adapter file is missing — re-run Step 10 (asset download) and Step 13 (validation)",
    "cannot import name 'apply_emotion_lora_to_model'": "train_lora.py was not patched — re-run Step 12 (emotion injection)",
    "anchor found": "an emotion patch anchor no longer matches the repo — the upstream repo changed; refresh the Step 12 patches",
}


def wait_for_healthy(timeout=STARTUP_TIMEOUT_SEC, poll_interval=POLL_INTERVAL_SEC):
    start = time.time()
    last_print = 0.0
    while time.time() - start < timeout:
        if live_proc.poll() is not None:
            tail_log(120)
            raise RuntimeError(
                f"live server process exited early with code {live_proc.returncode}; see log tail above"
            )

        log_text = ""
        if os.path.exists(LOG_PATH):
            with open(LOG_PATH, "r", errors="ignore") as f:
                log_text = f.read()

        markers_ok = {label: (needle in log_text) for label, needle in HEALTHY_MARKERS.items()}
        port_ok = port_listening("127.0.0.1", PORT)

        if all(markers_ok.values()) and port_ok:
            print("[ok] live server is healthy")
            for label, ok in markers_ok.items():
                print(f"  [OK] {label}")
            print(f"  [OK] port {PORT} listening")
            return True

        if time.time() - last_print > 15:
            elapsed = int(time.time() - start)
            print(f"[..] waiting ({elapsed}s/{timeout}s) markers={markers_ok} port_listening={port_ok}")
            last_print = time.time()

        time.sleep(poll_interval)

    print("[error] timed out waiting for healthy startup; log tail:")
    tail_log(150)
    log_text = ""
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, "r", errors="ignore") as f:
            log_text = f.read()
    for needle, hint in ERROR_HINTS.items():
        if needle in log_text:
            print(f"[diagnosis] found '{needle}' in log -> {hint}")
    raise TimeoutError("Live server did not become healthy within the timeout; see diagnostics above")


wait_for_healthy()


## Step 18 — HTTP / Uvicorn confirmation

Final confirmation that the FastAPI/Uvicorn app is actually serving HTTP on
`0.0.0.0:{PORT}` (default `8998`), not just that the port is open. This only
checks `127.0.0.1` inside the pod — it does **not** prove the RunPod public
proxy URL works, since that depends on `PORT` being one of the ports actually
exposed by your pod template (see the note in the Parameters cell). This cell
prints the expected public URL from `RUNPOD_POD_ID` + `PORT` for convenience,
plus the emotion that is now baked into the running server.


In [ ]:
import urllib.error
import urllib.request

try:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=10) as resp:
        body = resp.read(2000).decode(errors="ignore")
        print(f"[ok] HTTP {resp.status} from http://127.0.0.1:{PORT}/")
        print(body[:500])
except urllib.error.URLError as e:
    tail_log(80)
    raise RuntimeError(f"HTTP check failed against http://127.0.0.1:{PORT}/: {e}")

with open(LOG_PATH, "r", errors="ignore") as f:
    final_log = f.read()
assert f"Uvicorn running on http://{HOST}:{PORT}" in final_log, "Uvicorn startup banner not found in log"
assert "emotion lora loaded" in final_log, "emotion LoRA load marker not found in log"

print()
print("=" * 72)
print("SUCCESS: PersonaPlex + IMTalker emotion live server (AH AudioPace) is running and healthy")
if AUTO_EMOTION_DETECTION:
    print(f"  Emotion: {EMOTION} (id={EMOTION_ID}) — startup default; updates automatically per utterance")
    print("  Auto-emotion: ENABLED — run Step 19 (SER install) and Step 20 (worker) to activate")
else:
    print(f"  Emotion: {EMOTION} (id={EMOTION_ID}) — fixed for this entire session")
    print("  Auto-emotion: DISABLED (set AUTO_EMOTION_DETECTION=True in Parameters to enable)")
print(f"  Emotion adapter: {EMOTION_ADAPTER_PATH}")
print(f"  Internal: http://{HOST}:{PORT}")
if PUBLIC_URL:
    print(f"  Public:   {PUBLIC_URL}")
    print(f"  (auto-built from RUNPOD_POD_ID={RUNPOD_POD_ID} and PORT={PORT} — confirm this exact")
    print(f"   port is listed under this pod's Connect -> HTTP Service on the RunPod dashboard;")
    print(f"   the proxy only forwards ports that were exposed when the pod was created/edited)")
else:
    print(f"  Open port {PORT} through your RunPod pod's proxy/port mapping to access the browser UI")
    print(f"  (RUNPOD_POD_ID was not auto-detected; set it in the Parameters cell for a printed URL)")
print(f"  Browser UI file: static/index_v3_binary_fullscreen_aj_nodrop.html")
print(f"  PID: {open(PID_PATH).read().strip()}   Log: {LOG_PATH}")
print("=" * 72)


## Step 19 — Install Speech Emotion Recognition (SER) model

Downloads and caches `r-f/wav2vec-english-speech-emotion-recognition` from
Hugging Face. The model (~360 MB) is a `wav2vec2-base` backbone fine-tuned on
7 emotion classes. It runs on the same RTX 5090 as the FM generator —
inference takes ~20–35 ms per utterance.

> Only needed when `AUTO_EMOTION_DETECTION = True`. Safe to skip otherwise.


In [ ]:
# Step 19 — Download / cache the SER model
# (safe to skip when AUTO_EMOTION_DETECTION = False)

if not AUTO_EMOTION_DETECTION:
    print('[skip] AUTO_EMOTION_DETECTION=False — skipping SER model download')
else:
    # Pin stable releases with --no-deps to protect pinned PyTorch 2.8.0
    run(
        f'{VENV_ACTIVATE} && pip install --no-deps "huggingface_hub==0.26.2" "transformers==4.44.2" "tokenizers==0.19.1"',
        retries=2, retry_delay=10,
    )
    assert_torch_pin()

    _ser_dl_lines = [
        'import os, sys\n',
        "os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')\n",
        'from transformers import AutoConfig, AutoFeatureExtractor, AutoModelForAudioClassification, AutoModelForCTC\n',
        'MODEL_ID = sys.argv[1]\n',
        "print(f'Downloading / verifying SER model: {MODEL_ID} ...')\n",
        'cfg = AutoConfig.from_pretrained(MODEL_ID)\n',
        'feat = AutoFeatureExtractor.from_pretrained(MODEL_ID)\n',
        'if getattr(cfg, "architectures", None) and "Wav2Vec2ForCTC" in cfg.architectures:\n',
        '    mdl = AutoModelForCTC.from_pretrained(MODEL_ID)\n',
        'else:\n',
        '    mdl = AutoModelForAudioClassification.from_pretrained(MODEL_ID)\n',
        "print('SER model labels:', list(mdl.config.id2label.values()))\n",
        "print(f'[ok] SER model cached: {MODEL_ID}')\n",
    ]
    _ser_dl_path = '/tmp/ser_download.py'
    with open(_ser_dl_path, 'w', encoding='utf-8') as _fh:
        _fh.writelines(_ser_dl_lines)
    run(
        f'{VENV_ACTIVATE} && python {_ser_dl_path} "{SER_MODEL_ID}"',
        retries=2, retry_delay=15,
    )
    print(f'[ok] SER model ready: {SER_MODEL_ID}')


## Step 20 — Start auto-emotion detection worker

Starts an independent background daemon process (executed with `VENV_PYTHON` in your virtual environment) that:
1. Connects to `/ws/conversation` over local WebSocket and decodes user audio in real time.
2. Identifies user speech utterances via energy-based Voice Activity Detection (VAD).
3. Classifies the emotion from the user's voice using the Speech Emotion Recognition (SER) model.
4. Dynamically pushes the detected emotion to the live server via `POST /api/emotion`, causing the avatar's facial expression to update on the very next 0.96s FM chunk.
5. **Logs every prediction in human-readable format to a separate log file (`auto_emotion_predictions.log`)** so you can easily track and verify which emotion the model detected from your voice.


In [ ]:
# Step 20 — Auto-emotion detection worker
# Re-run to restart; set STOP_AUTO_EMOTION = True and re-run to stop.

STOP_AUTO_EMOTION = False   # <- set True and re-run this cell to stop the worker

import os
import subprocess
import time
import json
import urllib.request


def stop_auto_emotion():
    if not os.path.exists(AUTO_EMOTION_PID_PATH):
        print("[info] no auto-emotion worker pid file found; nothing to stop")
        return
    try:
        with open(AUTO_EMOTION_PID_PATH, "r") as f:
            pid = f.read().strip()
        if pid:
            print(f"[stop] terminating auto-emotion worker pid {pid} ...")
            subprocess.run(["kill", "-9", pid], check=False)
    except Exception as e:
        print(f"[warn] error stopping worker: {e}")
    try:
        os.remove(AUTO_EMOTION_PID_PATH)
    except OSError:
        pass
    print("[ok] auto-emotion worker stopped")


if not AUTO_EMOTION_DETECTION:
    print('[skip] AUTO_EMOTION_DETECTION=False — auto-emotion worker not started.')
    print('       Set AUTO_EMOTION_DETECTION=True in Parameters to enable.')
elif STOP_AUTO_EMOTION:
    stop_auto_emotion()
else:
    # 1. Stop previous worker instance if running
    stop_auto_emotion()

    # 2. Check if live server is reachable
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/emotion', timeout=5) as _r:
            _chk = json.loads(_r.read().decode('utf-8'))
        print(f'[auto-emotion] /api/emotion check OK: {_chk}')
    except Exception as _exc:
        raise RuntimeError(
            f'Cannot reach http://127.0.0.1:{PORT}/api/emotion: {_exc}\n'
            'Ensure the live server is running (Step 16) before launching the worker.'
        )

    # 3. Generate standalone worker script
    WORKER_SCRIPT = os.path.join(PROJECT_ROOT, "scripts", "auto_emotion_worker.py")
    os.makedirs(os.path.dirname(WORKER_SCRIPT), exist_ok=True)
    worker_src = '"""\nAuto-Emotion Detection Worker for PersonaPlex / IMTalker Live Avatar.\nListens to the user\'s voice on /ws/conversation, runs Speech Emotion Recognition (SER),\nupdates the avatar facial expression via POST /api/emotion, and logs predictions\nto a dedicated log file.\n"""\nimport sys\nimport os\nimport time\nimport json\nimport asyncio\nimport urllib.request\nimport numpy as np\n\nPORT = int(sys.argv[1]) if len(sys.argv) > 1 else int(os.environ.get("PORT", 8000))\nSER_MODEL_ID = sys.argv[2] if len(sys.argv) > 2 else os.environ.get("SER_MODEL_ID", "r-f/wav2vec-english-speech-emotion-recognition")\nLOG_PATH = sys.argv[3] if len(sys.argv) > 3 else os.environ.get("AUTO_EMOTION_LOG_PATH", "/workspace/speech2avatar/auto_emotion_predictions.log")\n\nMIN_UTTERANCE_SEC = float(os.environ.get("SER_MIN_UTTERANCE_SEC", 0.6))\nSILENCE_RMS_THRESHOLD = float(os.environ.get("SER_SILENCE_RMS_THRESHOLD", 0.015))\nUTTERANCE_END_SILENCE_SEC = float(os.environ.get("SER_UTTERANCE_END_SILENCE_SEC", 0.9))\nMIN_CONFIDENCE = float(os.environ.get("SER_MIN_CONFIDENCE", 0.35))\n\nEMOTION_MAP = {"neutral": 0, "happy": 1, "sad": 2, "angry": 3, "surprise": 4, "fear": 5}\nSER_LABEL_MAP = {\n    "angry": "angry",\n    "disgust": "sad",\n    "fear": "fear",\n    "happy": "happy",\n    "neutral": "neutral",\n    "sad": "sad",\n    "surprise": "surprise",\n    "calm": "neutral",\n    "fearful": "fear",\n    "surprised": "surprise",\n}\n\nTARGET_SR = 24000\n\n\ndef write_log(message: str, banner: bool = False):\n    """Write timestamped entry to both stdout and the dedicated log file."""\n    ts = time.strftime("%Y-%m-%d %H:%M:%S")\n    if banner:\n        formatted = f"\\n{\'=\'*80}\\n[{ts}] {message}\\n{\'=\'*80}\\n"\n    else:\n        formatted = f"[{ts}] {message}\\n"\n    print(formatted, end="", flush=True)\n    try:\n        os.makedirs(os.path.dirname(os.path.abspath(LOG_PATH)), exist_ok=True)\n        with open(LOG_PATH, "a", encoding="utf-8") as f:\n            f.write(formatted)\n            f.flush()\n    except Exception as e:\n        print(f"[warn] Failed writing to log file {LOG_PATH}: {e}", flush=True)\n\n\ndef push_emotion(port: int, emotion_id: int):\n    payload = json.dumps({"emotion_id": emotion_id}).encode("utf-8")\n    req = urllib.request.Request(\n        f"http://127.0.0.1:{port}/api/emotion",\n        data=payload,\n        headers={"Content-Type": "application/json"},\n        method="POST",\n    )\n    with urllib.request.urlopen(req, timeout=3) as resp:\n        return json.loads(resp.read().decode("utf-8"))\n\n\ndef get_current_emotion(port: int):\n    req = urllib.request.Request(f"http://127.0.0.1:{port}/api/emotion", method="GET")\n    with urllib.request.urlopen(req, timeout=3) as resp:\n        return json.loads(resp.read().decode("utf-8"))\n\n\ndef load_model():\n    import torch\n    from transformers import AutoConfig, AutoFeatureExtractor, AutoModelForAudioClassification, AutoModelForCTC\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    write_log(f"Loading SER model \'{SER_MODEL_ID}\' on {device} ...")\n\n    cfg = AutoConfig.from_pretrained(SER_MODEL_ID)\n    extractor = AutoFeatureExtractor.from_pretrained(SER_MODEL_ID)\n\n    if getattr(cfg, "architectures", None) and "Wav2Vec2ForCTC" in cfg.architectures:\n        model = AutoModelForCTC.from_pretrained(SER_MODEL_ID)\n    else:\n        model = AutoModelForAudioClassification.from_pretrained(SER_MODEL_ID)\n\n    model = model.to(device).eval()\n    labels = list(model.config.id2label.values())\n    write_log(f"SER model loaded successfully. Classes ({len(labels)}): {labels}")\n    return extractor, model, device\n\n\ndef classify_utterance(pcm_f32: np.ndarray, sr: int, extractor, model, device):\n    import torch\n    import torchaudio\n\n    wav = torch.from_numpy(pcm_f32).unsqueeze(0)\n    if sr != 16000:\n        wav = torchaudio.functional.resample(wav, sr, 16000)\n\n    inputs = extractor(wav.squeeze(0).numpy(), sampling_rate=16000, return_tensors="pt", padding=True)\n    inputs = {k: v.to(device) for k, v in inputs.items()}\n\n    with torch.no_grad():\n        outputs = model(**inputs)\n\n    logits = outputs.logits\n    if logits.ndim == 3:\n        logits = logits.mean(dim=1)\n\n    probs = torch.softmax(logits, dim=-1)[0]\n    pred_idx = int(torch.argmax(probs).item())\n    raw_label = model.config.id2label[pred_idx].lower().strip()\n    confidence = float(probs[pred_idx].item())\n\n    mapped_emotion = SER_LABEL_MAP.get(raw_label, "neutral")\n    mapped_id = EMOTION_MAP.get(mapped_emotion, 0)\n    return mapped_emotion, mapped_id, confidence, raw_label\n\n\nasync def run_worker():\n    import sphn\n    import websockets\n\n    write_log("Starting Auto-Emotion Worker daemon ...", banner=True)\n    write_log(f"Target Server Port  : {PORT}")\n    write_log(f"SER Model           : {SER_MODEL_ID}")\n    write_log(f"Predictions Log File: {LOG_PATH}")\n    write_log(f"Min Utterance       : {MIN_UTTERANCE_SEC}s")\n    write_log(f"Silence Threshold   : RMS >= {SILENCE_RMS_THRESHOLD}")\n    write_log(f"Utterance End Wait  : {UTTERANCE_END_SILENCE_SEC}s")\n    write_log(f"Min Confidence      : {MIN_CONFIDENCE * 100:.0f}%")\n\n    extractor, model, device = load_model()\n\n    try:\n        init_info = get_current_emotion(PORT)\n        current_eid = init_info.get("emotion_id", 0)\n        current_name = init_info.get("emotion_name", "neutral")\n        write_log(f"Initial avatar emotion: {current_name.upper()} (id={current_eid})")\n    except Exception as exc:\n        write_log(f"[warn] Could not fetch initial emotion: {exc}; default to neutral (id=0)")\n        current_eid = 0\n        current_name = "neutral"\n\n    silence_needed_frames = max(1, int(UTTERANCE_END_SILENCE_SEC * TARGET_SR / 1920))\n    loop = asyncio.get_running_loop()\n    uri = f"ws://127.0.0.1:{PORT}/ws/conversation"\n    backoff = 2.0\n\n    while True:\n        try:\n            write_log(f"Connecting to live audio stream at {uri} ...")\n            async with websockets.connect(\n                uri, max_size=10_000_000, open_timeout=10, ping_interval=20, ping_timeout=20,\n            ) as ws:\n                hello = json.loads(await ws.recv())\n                sess_id = hello.get("session_id", "unknown")[:8]\n                write_log(f"Connected to conversation session: {sess_id}. Listening for user speech ...")\n                await ws.send(json.dumps({"type": "start", "sample_rate": TARGET_SR}))\n\n                opus_reader = sphn.OpusStreamReader(TARGET_SR)\n                utt_buf = []\n                silence_frames = 0\n                backoff = 2.0\n\n                async for msg in ws:\n                    if not isinstance(msg, bytes) or len(msg) < 2 or msg[0] != 1:\n                        continue\n\n                    decoded = opus_reader.append_bytes(bytes(msg[1:]))\n                    if decoded.shape[-1] == 0:\n                        continue\n\n                    pcm = np.asarray(decoded, dtype=np.float32).reshape(-1)\n                    rms = float(np.sqrt(np.mean(np.square(pcm)))) if pcm.size else 0.0\n\n                    if rms >= SILENCE_RMS_THRESHOLD:\n                        utt_buf.append(pcm)\n                        silence_frames = 0\n                    else:\n                        if utt_buf:\n                            utt_buf.append(pcm)\n                            silence_frames += 1\n\n                        if silence_frames >= silence_needed_frames and utt_buf:\n                            full_pcm = np.concatenate(utt_buf, axis=0)\n                            utt_buf = []\n                            silence_frames = 0\n                            duration = len(full_pcm) / TARGET_SR\n\n                            if duration < MIN_UTTERANCE_SEC:\n                                continue\n\n                            mapped_emotion, mapped_id, conf, raw = await loop.run_in_executor(\n                                None, classify_utterance, full_pcm, TARGET_SR, extractor, model, device\n                            )\n\n                            pred_ts = time.strftime("%Y-%m-%d %H:%M:%S")\n                            log_lines = [\n                                "-" * 80,\n                                f"[{pred_ts}] USER VOICE DETECTED (duration: {duration:.2f}s, RMS: {rms:.4f})",\n                                f"  >>> PREDICTED EMOTION : {mapped_emotion.upper()} (id={mapped_id})",\n                                f"  >>> Model Label / Conf: \'{raw}\' ({conf * 100:.1f}%)",\n                            ]\n\n                            if conf < MIN_CONFIDENCE:\n                                log_lines.append(f"  >>> Action            : IGNORED (conf {conf:.2f} < threshold {MIN_CONFIDENCE:.2f}) -> Kept {current_name.upper()}")\n                                log_lines.append("-" * 80)\n                                write_log("\\n".join(log_lines))\n                                continue\n\n                            if mapped_id == current_eid:\n                                log_lines.append(f"  >>> Action            : UNCHANGED (avatar is already {current_name.upper()})")\n                                log_lines.append("-" * 80)\n                                write_log("\\n".join(log_lines))\n                                continue\n\n                            try:\n                                resp = await loop.run_in_executor(None, push_emotion, PORT, mapped_id)\n                                prev_name = current_name\n                                current_eid = mapped_id\n                                current_name = mapped_emotion\n                                log_lines.append(f"  >>> Action            : APPLIED -> Switched avatar from {prev_name.upper()} to {current_name.upper()} (id={current_eid})")\n                                log_lines.append(f"  >>> Server Response   : {resp}")\n                            except Exception as exc:\n                                log_lines.append(f"  >>> Action            : FAILED to update server: {exc}")\n\n                            log_lines.append("-" * 80)\n                            write_log("\\n".join(log_lines))\n\n        except Exception as exc:\n            write_log(f"Audio stream disconnected ({type(exc).__name__}: {exc}). Reconnecting in {backoff:.0f}s ...")\n            await asyncio.sleep(backoff)\n            backoff = min(backoff * 1.5, 30.0)\n\n\nif __name__ == "__main__":\n    try:\n        asyncio.run(run_worker())\n    except KeyboardInterrupt:\n        write_log("Auto-Emotion Worker interrupted by user. Exiting.")\n'
    with open(WORKER_SCRIPT, "w", encoding="utf-8") as f:
        f.write(worker_src)
    print(f"[ok] generated worker script: {WORKER_SCRIPT}")

    # 4. Prepare dedicated prediction log
    os.makedirs(os.path.dirname(AUTO_EMOTION_LOG_PATH), exist_ok=True)
    log_fp = open(AUTO_EMOTION_LOG_PATH, "a", encoding="utf-8")

    # 5. Launch worker as background daemon using VENV_PYTHON
    worker_env = os.environ.copy()
    worker_env.update({
        "PORT": str(PORT),
        "SER_MODEL_ID": SER_MODEL_ID,
        "AUTO_EMOTION_LOG_PATH": AUTO_EMOTION_LOG_PATH,
        "SER_MIN_UTTERANCE_SEC": str(SER_MIN_UTTERANCE_SEC),
        "SER_SILENCE_RMS_THRESHOLD": str(SER_SILENCE_RMS_THRESHOLD),
        "SER_UTTERANCE_END_SILENCE_SEC": str(SER_UTTERANCE_END_SILENCE_SEC),
        "SER_MIN_CONFIDENCE": str(SER_MIN_CONFIDENCE),
    })

    proc = subprocess.Popen(
        [VENV_PYTHON, WORKER_SCRIPT, str(PORT), SER_MODEL_ID, AUTO_EMOTION_LOG_PATH],
        cwd=PROJECT_ROOT, env=worker_env,
        stdout=log_fp, stderr=subprocess.STDOUT,
    )

    with open(AUTO_EMOTION_PID_PATH, "w") as f:
        f.write(str(proc.pid))

    print(f"[ok] launched auto-emotion worker: pid={proc.pid}")
    print(f"     dedicated prediction log : {AUTO_EMOTION_LOG_PATH}")
    print(f"     pid file                 : {AUTO_EMOTION_PID_PATH}")

    # 6. Verify process remains alive
    time.sleep(2.5)
    if proc.poll() is not None:
        print("[ERROR] worker process exited prematurely! Recent log output:")
        tail_auto_emotion_log(30)
        raise RuntimeError("Auto-emotion worker failed to start. See log output above.")

    print("\n[ok] Auto-Emotion Worker is running in the background! \u2713")
    print(f"  SER Model           : {SER_MODEL_ID}")
    print(f"  Silence RMS Thresh  : >= {SER_SILENCE_RMS_THRESHOLD}")
    print(f"  Utterance End Wait  : {SER_UTTERANCE_END_SILENCE_SEC}s")
    print(f"  Min Confidence      : {SER_MIN_CONFIDENCE * 100:.0f}%")
    print("  To view live predictions anytime, run: tail_auto_emotion_log(40)")
    print("  To stop worker: set STOP_AUTO_EMOTION = True and re-run this cell.")


## Operational cells (run any time)

These are safe to re-run independently after the server is up.


In [ ]:
# Tail the live server log
tail_log(200)


In [ ]:
# Tail the auto-emotion predictions log (shows detected emotions from user voice in real time)
tail_auto_emotion_log(40)


In [ ]:
# Full diagnostics — safe to run any time, including during a failure
def run_full_diagnostics():
    print("== GPU ==")
    run("nvidia-smi", check=False)

    print("\n== Torch / CUDA (venv) ==")
    try:
        print(get_torch_info())
    except Exception as e:
        print(f"[error] torch check failed: {e}")

    print(f"\n== Port {PORT} ==")
    print(f"listening: {port_listening('127.0.0.1', PORT)}  pids: {find_pids_on_port(PORT)}")

    print("\n== Required assets ==")
    for asset in REQUIRED_ASSETS:
        print(f"  [{'OK' if os.path.exists(asset['path']) else 'MISSING'}] {asset['path']}")
    print(f"  [{'OK' if os.path.exists(VOICE_PROMPT_PATH) else 'MISSING'}] {VOICE_PROMPT_PATH}")
    print(f"  [{'OK' if os.path.exists(EMOTION_ADAPTER_PATH) else 'MISSING'}] {EMOTION_ADAPTER_PATH}")

    print("\n== Emotion patch status ==")
    for path, needles in PATCH_CHECKS:
        text = path.read_text(encoding="utf-8") if path.exists() else ""
        missing = [n for n in needles if n not in text]
        print(f"  [{'OK' if not missing else 'FAIL'}] {path.name}" + (f" missing: {missing}" if missing else ""))

    print("\n== pip check (venv) ==")
    run(f"{VENV_ACTIVATE} && pip check", check=False)

    print("\n== Log tail ==")
    tail_log(100)


run_full_diagnostics()


In [ ]:
# Stop switch — set STOP_SERVER = True and re-run this cell to terminate the live server.
STOP_SERVER = False


def stop_server():
    if not os.path.exists(PID_PATH):
        print("[info] no pid file found; nothing to stop")
        return
    pid = open(PID_PATH).read().strip()
    print(f"[stop] terminating live server pid {pid}")
    subprocess.run(["kill", "-9", pid])
    os.remove(PID_PATH)
    print("[ok] server stopped")


if STOP_SERVER:
    stop_server()
else:
    print("STOP_SERVER is False — set it to True above and re-run this cell to stop the server.")


In [ ]:
# How to switch the avatar's emotion
# ─────────────────────────────────────────────────────────────────────────────
# AUTO-EMOTION mode (AUTO_EMOTION_DETECTION = True):
#   The emotion updates automatically per user utterance via Step 20.
#   No manual switching is needed. Run Step 20 to start/restart the worker.
#
# MANUAL mode (AUTO_EMOTION_DETECTION = False):
#   The emotion adapter weights are baked into the GPU model at startup, so
#   the EMOTION_ID cannot be switched live without a server restart.
#   To change it:
#     1. Run the STOP SERVER cell below and wait for the port to free.
#     2. In the Parameters cell, set:  EMOTION = "<new_emotion>"
#     3. Re-run the Parameters cell, then the Launch cell (Step 16).
#
# QUICK LIVE OVERRIDE (works in both modes — bypasses SER):
#   You can manually override the live emotion at any time by running:
#
#     import urllib.request, json
#     new_id = 2  # 0=neutral 1=happy 2=sad 3=angry 4=surprise 5=fear
#     req = urllib.request.Request(
#         f"http://127.0.0.1:{PORT}/api/emotion",
#         data=json.dumps({"emotion_id": new_id}).encode(),
#         headers={"Content-Type": "application/json"}, method="POST",
#     )
#     with urllib.request.urlopen(req) as r:
#         print(json.loads(r.read()))
# ─────────────────────────────────────────────────────────────────────────────

NEW_EMOTION = "sad"  # used only for the reference printout below

print(f"AUTO_EMOTION_DETECTION = {AUTO_EMOTION_DETECTION}")
print(f"Currently configured: EMOTION={EMOTION} (id={EMOTION_ID})")
print()
if AUTO_EMOTION_DETECTION:
    print("Auto-emotion mode is ON. The avatar's expression updates automatically.")
    print("Run Step 20 (auto-emotion worker) to start/restart the detection loop.")
    print()
    print("To manually override right now, POST to /api/emotion:")
    for _name, _id in EMOTION_MAP.items():
        print(f"  id={_id}: {_name}")
else:
    print(f"Manual mode. To switch to '{NEW_EMOTION}':")
    print(f"  1. Run the STOP SERVER cell and wait for the port to free")
    print(f"  2. In the Parameters cell, set:  EMOTION = \"{NEW_EMOTION}\"")
    print(f"  3. Re-run the Parameters cell, then the Launch cell (Step 16)")
    print()
    print("Emotion reference:")
    for _name, _id in EMOTION_MAP.items():
        marker = "  <-- active" if _id == EMOTION_ID else ""
        print(f"  {_id}: {_name}{marker}")
print()
print(f"Expression strength: EMOTION_CFG_SCALE={EMOTION_CFG_SCALE} (edit in Parameters;")
print("  raise for a stronger expression, lower toward 1.0 for subtler, then re-launch)")
